<p class="lead">Este notebook es la edición en R de la octava unidad del curso, en la que el perceptrón multicapa de la Unidad 6 se encuentra con los datos que hicieron famosas a las redes neuronales en medicina: las imágenes. Las diapositivas de clase son un mapa; este texto es el territorio. Cada figura y cada número que aparecen a continuación los produce código R que puedes ejecutar y modificar. Las secciones 4 y 5 entrenan perceptrones con imágenes, los evalúan con la metodología del Suplemento S1 y varían su topología —profundidad, ancho, activación, regularización, optimizador— en un diseño controlado que muestra qué mejora la clasificación y qué solo aparenta mejorarla. Las secciones 6 y 7 se preguntan <em>por qué</em> un perceptrón es un clasificador de imágenes poco adecuado, construyen a mano la operación de convolución y reproducen, en radiografías simuladas, la falla más documentada de la IA en imágenes médicas: una red que aprende el hospital en lugar de la enfermedad.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Explicar qué significa el valor de un píxel en cada una de las principales modalidades de imagen, representar una imagen como matriz y como vector de características, y describir el pipeline de reconocimiento de patrones de la Unidad 1 tal como se aplica a las imágenes.</li>
<li>Aplicar y justificar los pasos estándar de preprocesamiento —normalización de intensidades, ecualización del histograma, redimensionamiento, eliminación de ruido gaussiana y aumento de datos (data augmentation)— e indicar cuáles preservan la etiqueta de clase y cuáles no.</li>
<li>Calcular características de primer orden, de textura (co-ocurrencia de niveles de gris) y de forma a partir de una imagen, construir a mano la matriz de co-ocurrencia y entrenar con ellas los clasificadores de las Unidades 3–4 como línea base clásica.</li>
<li>Entrenar un perceptrón multicapa con imágenes aplanadas, evaluarlo con validación cruzada estratificada repetida, leer su matriz de confusión, visualizar los pesos de su primera capa y elegir un umbral de decisión para un contexto de tamizaje.</li>
<li>Diseñar y reportar un experimento honesto sobre la topología de la red: una grilla factorial evaluada con validación cruzada repetida, la varianza entre pliegues, curvas de aprendizaje y validación cruzada anidada para la estimación final.</li>
<li>Explicar la invariancia a la traslación, desarrollar a mano una convolución bidimensional y un paso de max-pooling, implementar ambos en R y mostrar por qué una etapa convolucional inicial es el puente entre el perceptrón y el aprendizaje profundo.</li>
<li>Reproducir el aprendizaje por atajos (shortcut learning) en datos de imagen, construir un mapa de sensibilidad a la oclusión que lo revele, y describir el desbalance de clases, el ruido en las etiquetas, la validación externa, los estudios con lectores y la situación regulatoria de la IA en imágenes.</li>
<li>Leer críticamente los artículos fundamentales de la IA en imágenes médicas: identificar la muestra, el estándar de referencia, la métrica, el comparador y lo que falta.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Léelo una vez de principio a fin, ejecutando cada bloque de código (chunk) a medida que llegas a él; luego vuelve a los experimentos de las secciones 5–7 y cambia un factor a la vez. Los ejercicios (cajas verdes) te piden modificar el bloque de código que está sobre ellos. Las cajas azules contienen ideas clave, las naranjas errores frecuentes y las moradas conectan esta unidad con otras unidades o con el curso de bioinformática (BGM324). El notebook usa solo R base y los paquetes habituales del curso —<code>nnet</code> y <code>neuralnet</code> para el perceptrón,<sup class="c"><a href="#r35">[35]</a></sup><sup class="c"><a href="#r36">[36]</a></sup> <code>e1071</code>, <code>randomForest</code> y <code>class</code> para la línea base clásica,<sup class="c"><a href="#r37">[37]</a></sup><sup class="c"><a href="#r38">[38]</a></sup> y <code>pROC</code> para la curva ROC—, sin ningún framework de aprendizaje profundo, porque el objetivo es entender las operaciones, no ejecutar una biblioteca. En un entorno de ejecución de R nuevo en Google Colab, el estudiante ejecuta una sola vez <code>install.packages(c("nnet","neuralnet","e1071","randomForest","pROC"))</code>. A lo largo del notebook se usan dos fuentes de imágenes, ambas <strong>simuladas</strong>: un conjunto de glifos sintéticos de 8 × 8 (seis formas geométricas) que cumple el papel del benchmark de dígitos manuscritos usado en la edición en Python,<sup class="c"><a href="#r10">[10]</a></sup> y un generador de parches de 16 × 16 similares a radiografías, escrito en la Sección 1.3. R no incluye ningún conjunto de datos de dígitos ni de imágenes médicas, así que estas fuentes los reemplazan; las imágenes están construidas para controlar el experimento —la posición, el tamaño y el contraste de cada estructura se conocen porque los elegimos nosotros— y no son radiografías reales. La Sección 1.3 enumera los conjuntos de datos reales que debes conocer.</p></div>



In [ ]:
# Bibliotecas usadas en todo el notebook. Las semillas están fijadas para que cada número citado en el texto sea reproducible.
suppressPackageStartupMessages({
  library(nnet); library(neuralnet); library(e1071); library(class)
  library(randomForest); library(pROC); library(MASS)
})
set.seed(0)
palette_pos <- "#d2691e"; palette_neg <- "#2b5d9b"; palette_acc <- "#3a7d44"
cat(R.version.string, "| nnet", as.character(packageVersion("nnet")),
    "| neuralnet", as.character(packageVersion("neuralnet")), "\n")



La salida impresa registra las versiones de R y de los paquetes con que se construyeron estas notas, el equivalente en R de imprimir las versiones de las bibliotecas al inicio de un notebook de Python. `set.seed(0)` hace reproducibles todas las particiones aleatorias y todas las imágenes simuladas que siguen.

## 1. Las imágenes médicas como patrones

### 1.1 Qué significa un píxel

Una imagen médica es una medición, y un clasificador que la lee hereda la física de esa medición. La revisión de Litjens y colaboradores, que catalogó más de 300 artículos de aprendizaje profundo sobre imágenes médicas hasta 2017, organiza el campo por modalidad precisamente por esta razón: lo que una red puede aprender depende de lo que codifican los números de la imagen.<sup class="c"><a href="#r1">[1]</a></sup> La tabla siguiente resume las modalidades que encontrarás en la literatura de esta unidad.

| Modalidad | Qué codifica el valor de un píxel (o vóxel) | Tamaño y profundidad de bits típicos | Tareas de clasificación en la literatura |
|---|---|---|---|
| **Radiografía de proyección** (rayos X) | Atenuación integrada de los rayos X a lo largo de la trayectoria del haz; el tejido denso (hueso, metal) atenúa más y por convención aparece brillante | 2,000–3,000 px por lado; 12–16 bits | neumonía, nódulos, fracturas, las 14 etiquetas torácicas de ChestX-ray14 |
| **Tomografía computarizada** (CT) | Coeficiente de atenuación lineal \( \mu \) de un pequeño volumen, expresado en **unidades Hounsfield** \( \mathrm{HU} = 1000\,(\mu - \mu_{\text{water}})/(\mu_{\text{water}} - \mu_{\text{air}}) \), de modo que el agua vale 0, el aire alrededor de −1000 y el hueso cortical alrededor de +1000<sup class="c"><a href="#r2">[2]</a></sup> | 512 × 512 por corte, cientos de cortes; 12 bits | nódulos pulmonares, hemorragia, neumonía por COVID-19, segmentación de órganos |
| **Resonancia magnética** (MRI) | Señal de los núcleos de hidrógeno tras la excitación por radiofrecuencia; depende de la densidad de protones, de los tiempos de relajación (T1, T2) y de la secuencia de pulsos — **unidades arbitrarias**, no comparables entre equipos sin normalización | 256 × 256 por corte; 12–16 bits | gradación de tumores cerebrales, lesiones de esclerosis múltiple, cáncer de próstata |
| **Ultrasonido** | Amplitud del eco devuelto por las interfaces de impedancia acústica, con ruido speckle intrínseco al método | 500–1,000 px por lado; 8 bits | lesiones mamarias, nódulos tiroideos, mediciones fetales |
| **Histopatología** (imágenes de lámina completa) | Color del tejido teñido bajo luz transmitida; tres canales (RGB); la tinción (H&E) es un contraste químico, no físico | 100,000 px por lado, procesadas en teselas; 8 bits por canal | detección de tumores, gradación, conteo de mitosis |
| **Dermatoscopía** | Luz reflejada por la superficie de la piel bajo aumento y polarización; RGB | 1,000–6,000 px por lado; 8 bits por canal | melanoma frente a nevo benigno (ISIC) |
| **Fotografía de retina** (fondo de ojo) | Luz reflejada por la retina a través de la pupila; RGB | 2,000–4,000 px por lado; 8 bits por canal | gradación de la retinopatía diabética, glaucoma |

De esto se siguen tres consecuencias. Los valores de CT son magnitudes físicas en una escala común, de modo que un umbral de −500 HU significa lo mismo en todos los equipos; los valores de MRI y de ultrasonido no lo son, por lo que la **normalización de intensidades** (Sección 2.1) es obligatoria antes de entrenar con imágenes de más de una máquina. Las imágenes clínicas tienen 12–16 bits de profundidad, pero se visualizan a través de una **ventana** (un rango de valores que se mapea a la escala de grises visible), de modo que un modelo entrenado con imágenes de 8 bits ventaneadas ve menos de lo que almacena la estación de trabajo. Y las imágenes se almacenan junto con sus parámetros de adquisición en el estándar **DICOM** (Digital Imaging and Communications in Medicine), cuyo encabezado registra el equipo, el protocolo, el espaciado de píxeles y, a menos que se elimine, la identidad del paciente: la desidentificación (Unidad 1) es una condición previa de todo conjunto de datos de imágenes, y los campos que sobreviven a ella —fabricante, institución— son precisamente los factores de confusión que, como muestra la Sección 7, una red puede explotar.

### 1.2 Imagen = matriz = vector

Una imagen con \( H \) filas y \( W \) columnas es una matriz \( \mathbf{I} \in \mathbb{R}^{H \times W} \) cuyo elemento \( I_{rc} \) es la intensidad del píxel en la fila \( r \) y la columna \( c \); una imagen en color agrega un tercer índice para el canal. Todos los clasificadores de las Unidades 2–6 esperan un *vector*, de modo que la matriz se aplana en \( \mathbf{x} = (I_{11}, I_{12}, \dots, I_{HW}) \) de longitud \( p = H \cdot W \). El aplanamiento es reversible y no pierde nada —pero descarta la *adyacencia* de los píxeles: al clasificador no se le dice que dos píxeles contiguos en la imagen son vecinos, mientras que dos que están a una fila de distancia también lo son. La Sección 6 muestra que esta es la debilidad central del perceptrón como clasificador de imágenes. La primera fuente de imágenes de la unidad es un conjunto de glifos sintéticos de 8 × 8 que se genera a continuación; el bloque de código imprime uno de ellos como la matriz que es.



In [ ]:
SG <- 8L                                   # lado del glifo (8 x 8, como el benchmark de dígitos de Python)
GLYPHS <- c("vbar","hbar","diagL","diagR","cross","ring")
draw_glyph <- function(cls){               # un glifo ruidoso como matriz SG x SG en [0,1]
  img <- matrix(0, SG, SG); gy <- matrix(rep(1:SG, SG), SG, SG); gx <- t(gy)
  ctr <- SG/2 + 0.5; cx <- ctr + round(runif(1,-1,1)); cy <- ctr + round(runif(1,-1,1)); th <- runif(1,0.7,1.1)
  if (cls=="vbar")  img[abs(gx-cx) <= th] <- 1
  else if (cls=="hbar")  img[abs(gy-cy) <= th] <- 1
  else if (cls=="diagL") img[abs((gx-cx)-(gy-cy)) <= th*1.3] <- 1
  else if (cls=="diagR") img[abs((gx-cx)+(gy-cy)) <= th*1.3] <- 1
  else if (cls=="cross") { img[abs(gx-cx) <= th] <- 1; img[abs(gy-cy) <= th] <- 1 }
  else if (cls=="ring")  { rr <- sqrt((gx-cx)^2+(gy-cy)^2); img[abs(rr-2.3) <= th] <- 1 }
  pmin(pmax(img*runif(1,0.5,0.9) + matrix(rnorm(SG*SG,0,0.33),SG,SG), 0), 1)
}
set.seed(7); g1 <- draw_glyph("cross")
cat("A 'cross' glyph as an 8 x 8 matrix (0 = background, 1 = bright):\n"); print(round(g1,2))
cat("Flattened:", round(as.vector(g1)[1:10],2), "...  length", length(g1), "\n")
par(mfrow=c(1,2), mar=c(4,4,2,1))
image(t(g1[SG:1,]), col=grey.colors(32, start=1, end=0), axes=FALSE, main="'cross' as an 8 x 8 matrix"); box()
barplot(as.vector(g1), col=palette_neg, border=NA, space=0, xlab="vector index", ylab="intensity",
        main="the same matrix flattened"); abline(v=(1:7)*SG*1.2, col=palette_pos, lty=2, lwd=0.6)



<p class="caption"><b>Figura 1.</b> Izquierda: un glifo sintético como la matriz de 8 × 8 números que el objeto realmente es; la imagen es solo una representación visual de esos números. Derecha: el vector de 64 elementos que se obtiene al leer la matriz columna por columna, el orden que usa <code>as.vector</code> de R (las líneas naranjas discontinuas marcan los límites entre columnas). Para el perceptrón, el último píxel de una columna y el primero de la siguiente no están más relacionados que cualquier otro par, aunque en la imagen puedan ser vecinos verticales. Los glifos reemplazan al benchmark de dígitos manuscritos de la edición en Python,<sup class="c"><a href="#r10">[10]</a></sup> que R no incluye.</p>

### 1.3 Parches de radiografía simulados, y los conjuntos de datos reales a los que reemplazan

Los glifos son imágenes pequeñas, pero no son médicas, y un notebook público de un curso no puede incluir radiografías. Por eso, la segunda fuente de imágenes de la unidad la genera una función escrita más abajo. Cada imagen es un **parche** de 16 × 16 en niveles de gris que imita una pequeña región de una radiografía: una textura de fondo suave, como de nubes (una suma de campos aleatorios a tres escalas espaciales, al estilo del *ruido de Perlin* usado en computación gráfica), una exposición global aleatoria y ruido aditivo en los píxeles. Un parche positivo contiene además un **nódulo** —una mancha más brillante, aproximadamente redonda, de posición, radio y contraste aleatorios— o un grupo de **microcalcificaciones**, un puñado de puntos brillantes muy pequeños. La posición de la lesión se sortea alrededor del centro del parche con un **desplazamiento aleatorio** (jitter) controlable, porque así es como llegan los parches a un clasificador en un sistema de detección asistida por computador: un detector de candidatos propone una ubicación y un clasificador de segunda etapa decide si el candidato es una lesión o un falso positivo, el diseño del desafío de nódulos pulmonares LUNA16.<sup class="c"><a href="#r11">[11]</a></sup> Por último, el generador puede estampar un pequeño rectángulo brillante en una esquina de los parches positivos de un "hospital A" ficticio —el tipo de marcador de lateralidad, regla o anotación que llevan las radiografías reales y que, como muestra la Sección 7, una red aprenderá en lugar de la lesión.

<div class="box warn"><h4>Estas son simulaciones, no radiografías</h4><p>Los parches siguientes se generan con una docena de líneas de código para que el experimento pueda ser <em>controlado</em>: conocemos la posición, el tamaño y el contraste de cada lesión, podemos producir tantas imágenes como necesitemos, y podemos introducir un factor de confusión y eliminarlo. Nada de eso es posible con datos reales, y nada de lo que se aprenda sobre <em>qué tan bien</em> un modelo clasifica estos parches se transfiere a las radiografías. Lo que sí se transfiere es el comportamiento de los métodos: qué hace el preprocesamiento, por qué el perceptrón es sensible a la posición, cómo se aprende un atajo y cómo se revela. Los conjuntos de datos reales que debes conocer, todos públicos y todos construidos con el esfuerzo de etiquetado y desidentificación descrito en sus artículos, son: <strong>ChestX-ray14</strong>, 112,120 radiografías frontales de tórax de 30,805 pacientes con 14 etiquetas extraídas de los informes mediante minería de texto;<sup class="c"><a href="#r3">[3]</a></sup> <strong>CheXpert</strong>, 224,316 radiografías de 65,240 pacientes con etiquetas que registran la incertidumbre;<sup class="c"><a href="#r4">[4]</a></sup> <strong>MIMIC-CXR</strong>, 377,110 imágenes de 227,835 estudios con sus informes en texto libre;<sup class="c"><a href="#r5">[5]</a></sup> <strong>CBIS-DDSM</strong>, un subconjunto curado y re-segmentado de la Digital Database for Screening Mammography con patología verificada;<sup class="c"><a href="#r6">[6]</a></sup> <strong>HAM10000</strong>, 10,015 imágenes dermatoscópicas de lesiones pigmentadas, el conjunto de entrenamiento de los desafíos de melanoma ISIC;<sup class="c"><a href="#r7">[7]</a></sup> y <strong>MedMNIST v2</strong>, una colección de 18 conjuntos de datos pequeños y estandarizados (imágenes de 28 × 28) que es el paso natural después de este notebook.<sup class="c"><a href="#r8">[8]</a></sup> Recuerda también que los datos de cáncer de mama de Wisconsin usados a lo largo del curso son ellos mismos <em>derivados de imágenes</em>: las 30 características se calcularon a partir de micrografías digitalizadas de núcleos celulares mediante un programa de segmentación semiautomática,<sup class="c"><a href="#r9">[9]</a></sup> lo que constituye el pipeline de características diseñadas a mano de la Sección 3 aplicado en 1993.</p></div>



In [ ]:
# Un generador reutilizable de parches simulados similares a radiografías (NO son imágenes reales; ver la caja anterior).
SIZE <- 16L
up_mat <- function(size, sc){                       # matriz de pesos separable para sobremuestreo bilineal
  xi <- seq(1, sc+1, length.out=size); x0 <- floor(xi); x1 <- pmin(x0+1, sc+1); fx <- xi-x0
  W <- matrix(0, size, sc+1)
  for (r in 1:size){ W[r,x0[r]] <- W[r,x0[r]]+(1-fx[r]); W[r,x1[r]] <- W[r,x1[r]]+fx[r] }; W
}
SCALES <- c(2,4,8); UM <- lapply(SCALES, function(sc) up_mat(SIZE, sc))
smooth_bg <- function(){                             # textura como de nubes: grillas aleatorias a 3 escalas, sumadas
  out <- matrix(0, SIZE, SIZE)
  for (k in seq_along(SCALES)){ sc <- SCALES[k]; g <- matrix(rnorm((sc+1)^2), sc+1, sc+1)
    out <- out + (1/sc) * (UM[[k]] %*% g %*% t(UM[[k]])) }
  out
}
make_patches <- function(n, lesion="nodule", jitter=3, hospital="B", marker_rate=0, positive=NULL,
                         contrast=c(0.3,0.6)){
  X <- array(0, c(n, SIZE, SIZE)); y <- integer(n)
  gy <- matrix(rep(1:SIZE, SIZE), SIZE, SIZE); gx <- t(gy)
  for (i in 1:n){
    img <- smooth_bg(); img <- (img-mean(img))/(sd(img)+1e-9)*0.10 + 0.45 + runif(1,-0.08,0.08)
    pos <- if (is.null(positive)) (runif(1)<0.5) else as.logical(positive)
    cx <- SIZE/2+0.5 + runif(1,-jitter,jitter); cy <- SIZE/2+0.5 + runif(1,-jitter,jitter)
    if (pos && lesion=="nodule"){ r <- runif(1,1.5,3); cc <- runif(1,contrast[1],contrast[2])
      img <- img + cc*exp(-((gx-cx)^2+(gy-cy)^2)/(2*(r/2)^2)) }
    else if (pos && lesion=="microcalc") for (j in 1:sample(3:6,1)){
      px <- min(max(round(cx+rnorm(1,0,1.3)),1),SIZE); py <- min(max(round(cy+rnorm(1,0,1.3)),1),SIZE)
      img[py,px] <- img[py,px] + runif(1,0.35,0.6) }
    if (hospital=="A" && pos && runif(1)<marker_rate) img[2:3,(SIZE-3):(SIZE-1)] <- 1.0   # marcador en la esquina
    img <- img + matrix(rnorm(SIZE*SIZE,0,0.03), SIZE, SIZE)
    X[i,,] <- pmin(pmax(img,0),1); y[i] <- as.integer(pos)
  }
  list(X=X, y=y)
}
set.seed(0);  nod <- make_patches(1000, "nodule")
set.seed(1);  mc  <- make_patches(1000, "microcalc")
cat(sprintf("nodule set: %d x %dx%d | positives %d | mean intensity neg %.3f, pos %.3f\n",
            dim(nod$X)[1], SIZE, SIZE, sum(nod$y), mean(nod$X[nod$y==0,,]), mean(nod$X[nod$y==1,,])))
cat(sprintf("microcalcification set: positives %d\n", sum(mc$y)))



Las dos intensidades medias impresas de las clases de nódulos difieren solo en el tercer decimal, lo cual es deliberado: un clasificador que mire solo el brillo promedio no puede resolver esta tarea, como confirman la Figura 2 y la Sección 3.



In [ ]:
set.seed(5); hA <- make_patches(6, "nodule", hospital="A", marker_rate=1.0)
show <- function(m, ...) image(t(m[SIZE:1,]), col=grey.colors(32,1,0), axes=FALSE, zlim=c(0,1), ...)
rows <- list("no lesion"=which(nod$y==0)[1:6], "nodule"=which(nod$y==1)[1:6],
             "microcalc."=which(mc$y==1)[1:6])
par(mfrow=c(4,6), mar=c(0.4,0.4,1.6,0.4), oma=c(0,7,2,0))
for (nm in names(rows)) for (j in seq_len(6)){ src <- if (nm=="microcalc.") mc$X else nod$X
  show(src[rows[[nm]][j],,]); if (j==1) mtext(nm, side=2, line=1, las=1, cex=0.7) }
for (j in 1:6){ show(hA$X[j,,]); if (j==1) mtext("hospital A\n(marker)", side=2, line=1, las=1, cex=0.7) }
mtext("Simulated 16 x 16 patches (the generator above; not real radiographs)", outer=TRUE, cex=0.9, font=2)



<p class="caption"><b>Figura 2.</b> Seis ejemplos de cada tipo de parche simulado. Los nódulos (segunda fila) varían en posición, tamaño y contraste, y algunos son apenas más brillantes que la textura de fondo: la tarea, deliberadamente, no es trivial. Los grupos de microcalcificaciones (tercera fila) son pequeños, brillantes y dispersos, un patrón que las estadísticas de primer orden capturan mal pero que los filtros locales capturan bien (Sección 6). La última fila muestra el hospital A ficticio, cuyos parches positivos llevan un rectángulo brillante en la esquina superior derecha; todo lo demás en ellos lo genera el mismo código. Como las dos clases de nódulos tienen una intensidad media casi idéntica, el brillo promedio por sí solo no resolverá la tarea.</p>

### 1.4 El pipeline de la Unidad 1, aplicado a imágenes

La Unidad 1 describió todo sistema de reconocimiento de patrones como sensado, preprocesamiento y segmentación, extracción de características, clasificación y decisión. Para las imágenes, las etapas son:

1. **Adquisición** — la física de la Sección 1.1 produce un archivo DICOM; un modelo entrenado con un protocolo no está validado para otro.
2. **Preprocesamiento** (Sección 2) — normalizar las intensidades, remuestrear a un tamaño común, suprimir el ruido y, durante el entrenamiento, aumentar los datos con transformaciones que preservan la etiqueta.
3. **Segmentación** — aislar la estructura de interés mediante umbralización y componentes conexas (Sección 3.3) o, en la práctica moderna, mediante una red que etiqueta cada píxel (fuera del alcance de esta unidad).
4. **Extracción de características** (Sección 3) — calcular un vector de longitud fija: estadísticas de primer orden, textura, forma. En una red neuronal (secciones 4–6) esta etapa se *aprende*.
5. **Clasificación y decisión** (secciones 4–5, 7) — un clasificador mapea el vector a una probabilidad, y un umbral elegido para el contexto clínico mapea la probabilidad a una acción.

La distinción entre las etapas 4 y 5 organiza la unidad: la Sección 3 las mantiene separadas, como lo hizo el campo hasta alrededor de 2012; la Sección 4 las fusiona en el perceptrón; la Sección 6 muestra que la fusión funciona mucho mejor cuando la arquitectura respeta la estructura de las imágenes.

## 2. Preprocesamiento

El preprocesamiento transforma cada imagen con la misma regla antes de que ocurra cualquier aprendizaje, para eliminar la variación que *no* es informativa sobre la clase —ganancia del equipo, exposición, resolución, ruido— de modo que la capacidad del clasificador se invierta en la variación que sí lo es. Cada operación siguiente se define, se implementa en pocas líneas de R base y se muestra sobre un glifo y sobre un parche sintético.

### 2.1 Normalización de intensidades y ecualización del histograma

Dos normalizaciones lineales son universales. El **escalamiento min–max** lleva la imagen a \( [0,1] \): \( I' = (I - I_{\min})/(I_{\max} - I_{\min}) \); elimina el desplazamiento y la ganancia, pero queda dominado por un único píxel extremo. La **estandarización** (z-score), \( I' = (I - \mu_I)/\sigma_I \), es el análogo por imagen de la estandarización por columnas que usas desde la Unidad 3 y es la opción habitual en MRI, donde los valores absolutos carecen de significado. En CT, donde los valores son físicos, la operación estándar es en cambio el **ventaneo** (windowing): los valores bajo un límite inferior se recortan a negro, los valores sobre un límite superior a blanco, y el intervalo intermedio se estira linealmente; una "ventana pulmonar" de aproximadamente −1,000 a +400 HU muestra el parénquima, una "ventana ósea" de aproximadamente −500 a +1,300 HU muestra el esqueleto, y el mismo corte de CT se ve completamente distinto a través de ambas.

La **ecualización del histograma** es no lineal: reasigna las intensidades para que el histograma de la salida sea lo más plano posible, lo que distribuye los niveles de gris de una imagen de bajo contraste por todo el rango. El algoritmo cabe en tres líneas. Para una imagen con \( L \) niveles de gris, calcula el histograma \( h(v) \), su distribución acumulada \( F(v) = \sum_{u \le v} h(u)/(HW) \), y asigna a cada píxel el valor \( v' = F(v) \). El bloque de código implementa las tres operaciones en R base y las aplica a un glifo y a un parche de bajo contraste.



In [ ]:
minmax  <- function(m) (m - min(m)) / (max(m) - min(m) + 1e-12)
zscore  <- function(m) (m - mean(m)) / (sd(m) + 1e-12)
equalise <- function(m, levels=256){                    # ecualización del histograma mediante la distribución acumulada
  q   <- pmin(round(minmax(m) * (levels-1)), levels-1)
  cdf <- cumsum(tabulate(q+1, levels)) / length(q)
  matrix(cdf[q+1], nrow(m), ncol(m))
}
low <- 0.25 + 0.25 * nod$X[which(nod$y==1)[4],,]          # un parche positivo con el contraste reducido a un cuarto
set.seed(7); dig <- draw_glyph("ring")
cat(sprintf("glyph: range [%.2f, %.2f] mean %.2f sd %.2f | after z-score: mean %.2f sd %.2f\n",
            min(dig), max(dig), mean(dig), sd(dig), mean(zscore(dig)), sd(zscore(dig))))
cat(sprintf("low-contrast patch: range [%.3f, %.3f] -> min-max [%.1f, %.1f] -> equalised sd %.3f (was %.3f)\n",
            min(low), max(low), min(minmax(low)), max(minmax(low)), sd(equalise(low)), sd(low)))
par(mfcol=c(2,4), mar=c(3,3,2,1))
for (pair in list(list(dig,"glyph"), list(low,"low-contrast patch"))){ im <- pair[[1]]; nm <- pair[[2]]
  image(t(im[nrow(im):1,]), col=grey.colors(32,1,0), axes=FALSE, main=paste(nm,"original")); box()
  image(t(minmax(im)[nrow(im):1,]), col=grey.colors(32,1,0), axes=FALSE, main="min-max to [0,1]"); box()
  image(t(equalise(im)[nrow(im):1,]), col=grey.colors(32,1,0), axes=FALSE, main="histogram-equalised"); box()
  hist(minmax(im), breaks=16, col=adjustcolor(palette_neg,0.6), border=NA, main="histograms", xlab="intensity")
  hist(equalise(im), breaks=16, col=adjustcolor(palette_pos,0.5), border=NA, add=TRUE) }



<p class="caption"><b>Figura 3.</b> De arriba abajo dentro de cada par: la imagen original, su versión min–max, su versión ecualizada y los dos histogramas (azul min–max, naranja ecualizado). El escalamiento min–max restituye el rango de visualización, pero no la distribución: el histograma del parche de bajo contraste sigue siendo un pico estrecho. La ecualización distribuye los niveles de gris de modo que cada uno se use aproximadamente con la misma frecuencia, lo que hace visible el nódulo y multiplica varias veces la desviación estándar, como se imprime. En el glifo, donde la mayoría de los píxeles son fondo, la ecualización sobre todo exagera el ruido. La ecualización es una operación global; la versión adaptativa con contraste limitado (CLAHE), aplicada por teselas, es la que se usa en la práctica en radiografías y en el preprocesamiento de los conjuntos de datos de retina y dermatoscopía de la Sección 8.</p>

<div class="box warn"><h4>¿Normalizar por imagen o por conjunto de datos?</h4><p>La normalización por imagen (esta sección) elimina las diferencias de exposición entre imágenes; la estandarización por característica sobre el conjunto de entrenamiento pone cada posición de píxel en la misma escala para el optimizador. Son operaciones distintas y normalmente se aplican ambas. Se haga lo que se haga, los parámetros de cualquier normalización <em>a nivel del conjunto de datos</em> —las medias y desviaciones estándar de las columnas— deben calcularse solo con los pliegues de entrenamiento y aplicarse al pliegue de prueba, exactamente como exigía el Suplemento S1 para evitar la fuga de datos. En este notebook, todo modelo con validación cruzada estandariza dentro del pliegue (la función auxiliar <code>sc_fit</code> de la Sección 3.4 recalcula el centro y la escala con las filas de entrenamiento), lo que equivale en R a ajustar un escalador solo con la partición de entrenamiento.</p></div>

### 2.2 Redimensionamiento y eliminación de ruido

Las redes necesitan entradas de tamaño fijo, y las imágenes llegan con muchos tamaños y espaciados de píxel. El **redimensionamiento** por interpolación cambia el número de píxeles; el orden de interpolación importa: el vecino más cercano conserva bloques, mientras que la interpolación bilineal produce bordes suaves que nunca se midieron. Reducir el tamaño promediando bloques de píxeles (**submuestreo**, downsampling) es la operación que realizan las capas de pooling en la Sección 6. La **eliminación de ruido** con un filtro gaussiano reemplaza cada píxel por un promedio ponderado de sus vecinos con pesos \( \propto \exp(-d^2/2\sigma^2) \); es una convolución (Sección 6.2), y su único parámetro \( \sigma \) equilibra ruido frente a nitidez. El bloque de código implementa cada operación en R base y las aplica a un glifo y a un parche al que se le agregó ruido adicional.



In [ ]:
zoom_nn <- function(m, k){ H<-nrow(m); m[rep(1:H, each=k), rep(1:ncol(m), each=k)] }      # vecino más cercano
zoom_bl <- function(m, size){ W <- up_mat(size, nrow(m)-1); W %*% m %*% t(W) }            # bilineal mediante up_mat
block_mean <- function(m, s){ H<-nrow(m)/s; W<-ncol(m)/s                                    # submuestreo por media de bloques s x s
  outer(1:H,1:W, Vectorize(function(r,c) mean(m[((r-1)*s+1):(r*s),((c-1)*s+1):(c*s)]))) }
gkern <- function(sigma, sz){ r <- -(sz%/%2):(sz%/%2); g <- exp(-(outer(r^2,r^2,"+"))/(2*sigma^2)); g/sum(g) }
conv_same <- function(I, K){ k<-nrow(K); p<-k%/%2; H<-nrow(I); W<-ncol(I)                   # correlación con relleno "same"
  Ip <- matrix(0, H+2*p, W+2*p); Ip[(p+1):(p+H),(p+1):(p+W)] <- I; O <- matrix(0,H,W)
  for (r in 1:H) for (c in 1:W) O[r,c] <- sum(K * Ip[r:(r+k-1), c:(c+k-1)]); O }
gauss_filter <- function(m, sigma) conv_same(m, gkern(sigma, 2*ceiling(2*sigma)+1))
set.seed(2); patch <- nod$X[which(nod$y==1)[5],,]; noisy <- pmin(pmax(patch + rnorm(SIZE^2,0,0.12),0),1)
mae <- function(a,b) mean(abs(a-b))
cat(sprintf("mean abs. error vs clean patch: noisy %.3f | sigma 0.5 %.3f | 1.0 %.3f | 2.0 %.3f\n",
            mae(noisy,patch), mae(gauss_filter(noisy,0.5),patch), mae(gauss_filter(noisy,1),patch),
            mae(gauss_filter(noisy,2),patch)))
par(mfrow=c(2,5), mar=c(1,1,2,1))
set.seed(7); gl <- draw_glyph("diagR")
for (im in list(list(gl,"glyph 8x8"), list(zoom_nn(gl,4),"zoom x4 nearest"), list(zoom_bl(gl,32),"zoom x4 bilinear"),
                list(block_mean(zoom_bl(gl,32),2),"down to 16x16"), list(block_mean(patch,2),"patch to 8x8")))
  { image(t(im[[1]][nrow(im[[1]]):1,]), col=grey.colors(32,1,0), axes=FALSE, main=im[[2]], cex.main=0.8); box() }
for (im in list(list(patch,"clean patch"), list(noisy,"patch + noise"), list(gauss_filter(noisy,0.5),"Gaussian s=0.5"),
                list(gauss_filter(noisy,1),"Gaussian s=1"), list(gauss_filter(noisy,2),"Gaussian s=2")))
  { image(t(im[[1]][nrow(im[[1]]):1,]), col=grey.colors(32,1,0), axes=FALSE, main=im[[2]], cex.main=0.8); box() }



<p class="caption"><b>Figura 4.</b> Arriba: un glifo ampliado cuatro veces por interpolación de vecino más cercano y bilineal, y luego submuestreado; la versión de vecino más cercano es honesta respecto de la grilla de ocho por ocho, la versión bilineal inventa bordes suaves. Arriba a la derecha: un parche submuestreado a 8 × 8 promediando bloques de 2 × 2 —un cuarto de los píxeles, y el nódulo sigue visible. Abajo: el parche limpio, el mismo parche con mucho ruido agregado, y el parche ruidoso suavizado con filtros gaussianos de ancho creciente. Los errores absolutos medios impresos respecto del parche limpio bajan desde la imagen ruidosa hasta \( \sigma = 1 \) y vuelven a subir en \( \sigma = 2 \), donde el filtro empieza a borrar el propio nódulo: la eliminación de ruido tiene un óptimo, y ese óptimo depende del tamaño de la estructura que se quiere conservar.</p>

### 2.3 Aumento de datos

Un clasificador que ve cada imagen una sola vez, en una sola orientación, aprende la orientación además del patrón. El **aumento de datos** (data augmentation) amplía la muestra con copias transformadas de sus imágenes —volteos, rotaciones, desplazamientos, pequeños cambios de escala y de intensidad— con la condición de que la transformación **preserve la etiqueta**.<sup class="c"><a href="#r12">[12]</a></sup> La condición es todo el método: una radiografía de tórax volteada de izquierda a derecha sigue mostrando neumonía, pero un corazón en espejo es *situs inversus*, un diagnóstico en sí mismo; una imagen dermatoscópica puede rotarse en cualquier ángulo, porque la piel no tiene "arriba"; una barra vertical rotada en 90° se convierte en una barra horizontal, una clase distinta. El aumento de datos también implementa una **invariancia** en la que cree quien diseña el sistema —"la etiqueta no depende de un desplazamiento de un píxel"— y la Sección 6 mide cuánto de esa invariancia adquiere un perceptrón a partir de él.



In [ ]:
shift_img <- function(m, dx, dy){ H<-nrow(m); W<-ncol(m); o<-matrix(0,H,W)                 # desplazamiento entero, relleno con ceros
  for (r in 1:H) for (c in 1:W){ sr<-r-dy; sc<-c-dx; if (sr>=1&&sr<=H&&sc>=1&&sc<=W) o[r,c]<-m[sr,sc] }; o }
rotate_nn <- function(m, deg){ H<-nrow(m); ctr<-(H+1)/2; a<--deg*pi/180; o<-matrix(0,H,H) # rotación por vecino más cercano
  for (r in 1:H) for (c in 1:H){ sr<-round(ctr+(r-ctr)*cos(a)-(c-ctr)*sin(a)); sc<-round(ctr+(r-ctr)*sin(a)+(c-ctr)*cos(a))
    if (sr>=1&&sr<=H&&sc>=1&&sc<=H) o[r,c]<-m[sr,sc] }; o }
augment <- function(m, kind) switch(kind,
  "flip L-R"=m[,ncol(m):1], "flip U-D"=m[nrow(m):1,], "rotate 90"=t(m)[,nrow(m):1],
  "rotate 15"=rotate_nn(m,15), "shift 1px"=shift_img(m,1,1), "brighter"=pmin(m*1.3,1))
kinds <- c("flip L-R","flip U-D","rotate 90","rotate 15","shift 1px","brighter")
set.seed(3); vb <- draw_glyph("vbar"); pt <- nod$X[which(nod$y==1)[2],,]
par(mfrow=c(2,7), mar=c(1,1,2,1))
for (base in list(list(vb,"vbar glyph"), list(pt,"nodule patch"))){
  image(t(base[[1]][nrow(base[[1]]):1,]),col=grey.colors(32,1,0),axes=FALSE,main="original",cex.main=0.8); box()
  for (k in kinds){ image(t(augment(base[[1]],k)[nrow(base[[1]]):1,]),col=grey.colors(32,1,0),axes=FALSE,main=k,cex.main=0.8); box() } }
cat("Label preserved?  vbar glyph: flips/90-rotation NO (a flipped or turned bar is another class); small rotation/shift/brightness YES\n")
cat("                  patch: all six YES (a nodule is a nodule in any orientation and exposure)\n")



<p class="caption"><b>Figura 5.</b> Seis aumentos de un glifo (arriba) y de un parche con nódulo (abajo). Para el parche, toda transformación es admisible: el nódulo sigue siendo un nódulo. Para el glifo de barra vertical, los volteos y el cuarto de vuelta cambian la clase (una barra vertical girada un cuarto de vuelta es una barra horizontal), mientras que la rotación pequeña, el desplazamiento y el cambio de brillo no lo hacen. Por lo tanto, el aumento de datos requiere una decisión de alguien que conozca el dominio, y esa es una de las razones por las que un bioingeniero, y no solo un informático, tiene un lugar en un equipo de IA en imágenes. El aumento se aplica solo a los pliegues de entrenamiento; aumentar antes de dividir pondría copias transformadas de una imagen de prueba en el conjunto de entrenamiento, la fuga de datos del Suplemento S1 bajo un nuevo disfraz.</p>

<div class="exercise"><h4>Ejercicio 1</h4><p>En el bloque de código del aumento de datos, reemplaza la <code>vbar</code> por una <code>cross</code> y por un <code>ring</code> y decide, para cada transformación, si la etiqueta se preserva. (La <code>cross</code> y el <code>ring</code> son simétricos frente a los volteos y al cuarto de vuelta, de modo que para ellos son admisibles más transformaciones que para la barra: el conjunto admisible depende de la simetría de la clase.) Luego agrega una rotación de 180° (<code>rotate_nn(m, 180)</code>) y enumera las clases de glifos para las cuales preserva la etiqueta.</p></div>

## 3. Características diseñadas a mano frente a características aprendidas

Hasta alrededor de 2012, clasificar imágenes significaba diseñar características: números calculados a partir de la imagen mediante una regla escrita por el ingeniero, sugerida por la descripción que un radiólogo hace de la lesión —"más brillante", "redonda", "textura heterogénea"— y entregados a los clasificadores de las Unidades 3 y 4. Esta sección construye un pequeño extractor de ese tipo, desarrolla a mano su componente de textura y entrena con él k vecinos más cercanos, una máquina de vectores de soporte y un bosque aleatorio: la **línea base clásica** frente a la cual se juzga el perceptrón de la Sección 4.

### 3.1 Características de primer orden

Las características de primer orden dependen solo de la distribución de las intensidades, no de su disposición: la media, la desviación estándar, la asimetría, la curtosis y el rango de los valores de los píxeles, y la **entropía** del histograma, \( -\sum_v p(v) \log_2 p(v) \), que mide qué tan dispersos están los niveles de gris. Un nódulo aumenta ligeramente la asimetría (una cola brillante) y el máximo del parche; las características de textura capturan cómo está dispuesto el brillo.

### 3.2 Textura: la matriz de co-ocurrencia de niveles de gris, a mano

La matriz de co-ocurrencia de Haralick (GLCM) es el descriptor de textura clásico, todavía usado en radiómica.<sup class="c"><a href="#r13">[13]</a></sup> Cuantiza la imagen a \( L \) niveles de gris, elige un desplazamiento (por ejemplo, "un píxel a la derecha") y cuenta cuántas veces un píxel de nivel \( i \) tiene un vecino de nivel \( j \) en ese desplazamiento. Los conteos, normalizados para que sumen uno, forman la matriz \( L \times L \) \( P(i,j) \); a partir de ella se calculan luego características escalares. Tres de las catorce de Haralick bastan para esta unidad:

<div class="eq">\[ \text{contrast} = \sum_{i,j} (i-j)^2 P(i,j), \qquad \text{homogeneity} = \sum_{i,j} \frac{P(i,j)}{1+|i-j|}, \qquad \text{energy} = \sum_{i,j} P(i,j)^2 \]
<span class="where">El contraste es grande cuando los píxeles vecinos difieren (bordes, ruido); la homogeneidad es grande cuando se parecen (regiones suaves); la energía es grande cuando unas pocas transiciones dominan (textura regular). Una cuarta característica, la correlación entre \( i \) y \( j \) bajo \( P \), mide la estructura lineal.</span></div>

Toma una imagen de 4 × 4 cuantizada a \( L = 4 \) niveles y el desplazamiento "un píxel a la derecha":

\[ \mathbf{I} = \begin{pmatrix} 0 & 0 & 1 & 1 \\ 0 & 0 & 1 & 1 \\ 0 & 2 & 2 & 2 \\ 2 & 2 & 3 & 3 \end{pmatrix} \]

Cada fila tiene tres pares horizontales \( (I_{r,c}, I_{r,c+1}) \), doce en total: la fila 1 da (0,0), (0,1), (1,1); la fila 2 lo mismo; la fila 3 da (0,2), (2,2), (2,2); la fila 4 da (2,2), (2,3), (3,3). Contando:

| par \( (i,j) \) | (0,0) | (0,1) | (0,2) | (1,1) | (2,2) | (2,3) | (3,3) |
|---|---|---|---|---|---|---|---|
| conteo | 2 | 2 | 1 | 2 | 3 | 1 | 1 |
| \( P(i,j) \) | 2/12 | 2/12 | 1/12 | 2/12 | 3/12 | 1/12 | 1/12 |

Entonces, el contraste \( = [2 \cdot 1^2 + 1 \cdot 2^2 + 1 \cdot 1^2]/12 = 7/12 \approx 0.583 \) (solo contribuyen los pares fuera de la diagonal); la homogeneidad \( = [8 \cdot 1 + 2 \cdot \tfrac{1}{2} + 1 \cdot \tfrac{1}{3} + 1 \cdot \tfrac{1}{2}]/12 \approx 0.819 \); la energía \( = (4+4+1+4+9+1+1)/144 = 24/144 \approx 0.167 \). (La definición original de Haralick simetriza la matriz sumándole su transpuesta; la forma no simetrizada que se usa aquí da el mismo contraste y la misma homogeneidad y es más sencilla de seguir.) El bloque de código confirma la aritmética.



In [ ]:
glcm <- function(q, levels, off=c(0,1)){               # matriz de co-ocurrencia normalizada para un desplazamiento (dy,dx)
  dy <- off[1]; dx <- off[2]; H <- nrow(q); W <- ncol(q)
  a <- q[1:(H-dy), 1:(W-dx)]; b <- q[(1+dy):H, (1+dx):W]; P <- matrix(0, levels, levels)
  for (k in seq_along(a)) P[a[k]+1, b[k]+1] <- P[a[k]+1, b[k]+1] + 1
  P / sum(P)
}
glcm_feats <- function(img, levels=8, offs=list(c(0,1), c(1,0))){
  q <- pmin(floor(minmax(img)*levels), levels-1); idx <- 0:(levels-1)
  I <- matrix(idx, levels, levels); J <- t(I); out <- NULL
  for (off in offs){ P <- glcm(q, levels, off); mi <- sum(P*I); mj <- sum(P*J)
    si <- sqrt(sum(P*(I-mi)^2)); sj <- sqrt(sum(P*(J-mj)^2))
    out <- rbind(out, c(sum(P*(I-J)^2), sum(P/(1+abs(I-J))), sum(P^2), sum(P*(I-mi)*(J-mj))/(si*sj+1e-12))) }
  colMeans(out)
}
Ih <- matrix(c(0,0,1,1, 0,0,1,1, 0,2,2,2, 2,2,3,3), 4, 4, byrow=TRUE)
P <- glcm(Ih, 4, c(0,1)); i <- matrix(0:3,4,4); j <- t(i)
cat("Co-occurrence counts (rows i, cols j), displacement one pixel to the right:\n"); print(round(P*12))
cat(sprintf("contrast = %.3f   homogeneity = %.3f   energy = %.3f\n",
            sum(P*(i-j)^2), sum(P/(1+abs(i-j))), sum(P^2)))



Los conteos y las tres características coinciden con el cálculo a mano: contraste 0.583, homogeneidad 0.819, energía 0.167. La función `glcm_feats` cuantiza una imagen a ocho niveles y promedia las cuatro características sobre los desplazamientos horizontal y vertical, de modo que una textura rotada en un cuarto de vuelta produce los mismos números.

### 3.3 Características de forma a partir de una máscara umbralizada

La forma requiere una **segmentación**: una máscara binaria que indica qué píxeles pertenecen al objeto. La segmentación más simple es un umbral —aquí, los píxeles más brillantes que la media del parche más 1.5 desviaciones estándar tras un suavizado leve— seguido del **etiquetado de componentes conexas**, que agrupa los píxeles de primer plano adyacentes en objetos. El bloque de código implementa el etiquetado como un breve algoritmo de relleno por inundación (flood fill) en R base (el sustituto en R de `scipy.ndimage.label`). A partir de la componente más grande calculamos su área, su contraste respecto del fondo y su excentricidad (a partir de los valores propios de la covarianza de las coordenadas de sus píxeles: 0 para un disco, cercana a 1 para una línea); el número de componentes es una cuarta característica que distingue un nódulo de una dispersión de microcalcificaciones. Esta es una miniatura de lo que hizo el programa de Wisconsin con los núcleos celulares en 1993.<sup class="c"><a href="#r9">[9]</a></sup> El bloque de código reúne las catorce características y aplica el extractor a los mil parches de nódulos.



In [ ]:
label_cc <- function(mask){                              # componentes conexas con 4-conectividad (relleno por inundación)
  H <- nrow(mask); W <- ncol(mask); lab <- matrix(0, H, W); cur <- 0
  for (r in 1:H) for (c in 1:W) if (mask[r,c] && lab[r,c]==0){ cur <- cur+1; st <- list(c(r,c))
    while (length(st)>0){ p <- st[[length(st)]]; st[[length(st)]] <- NULL; pr <- p[1]; pc <- p[2]
      if (pr<1||pr>H||pc<1||pc>W) next; if (!mask[pr,pc]||lab[pr,pc]!=0) next; lab[pr,pc] <- cur
      st <- c(st, list(c(pr-1,pc),c(pr+1,pc),c(pr,pc-1),c(pr,pc+1))) } }
  list(lab=lab, n=cur)
}
shape_feats <- function(img){ sm <- gauss_filter(img, 1.0); mask <- sm > mean(sm) + 1.5*sd(sm)
  cc <- label_cc(mask); if (cc$n==0) return(c(0,0,0,0))
  sizes <- tabulate(cc$lab[cc$lab>0], cc$n); k <- which.max(sizes); comp <- cc$lab==k
  ij <- which(comp, arr.ind=TRUE)
  ev <- if (nrow(ij)>2) sort(eigen(cov(ij), only.values=TRUE)$values, decreasing=TRUE) else c(1,1)
  c(sizes[k], cc$n, sqrt(max(1 - ev[2]/(ev[1]+1e-9), 0)), mean(sm[comp]) - mean(sm[!mask])) }
FEATURE_NAMES <- c("mean","sd","skewness","kurtosis","range","entropy",
                   "glcm_contrast","glcm_homogeneity","glcm_energy","glcm_correlation",
                   "area","n_objects","eccentricity","obj_contrast")
extract_features <- function(img){ v <- as.vector(img)
  h <- tabulate(pmin(floor(v*32)+1, 32), 32) / length(v); h <- h[h>0]
  first <- c(mean(v), sd(v), e1071::skewness(v), e1071::kurtosis(v), max(v)-min(v), -sum(h*log2(h)))
  c(first, glcm_feats(img), shape_feats(img)) }
nod_F <- t(sapply(1:dim(nod$X)[1], function(k) extract_features(nod$X[k,,])))
colnames(nod_F) <- FEATURE_NAMES
cat("feature matrix:", nrow(nod_F), "x", ncol(nod_F), "\n")
print(round(rbind(`no lesion`=colMeans(nod_F[nod$y==0,]), nodule=colMeans(nod_F[nod$y==1,])), 3))



La tabla de medias por clase ya cuenta una historia. La media de primer orden apenas se mueve entre las clases, como anticipó la Figura 2, mientras que las características de forma y de distribución las separan: los parches positivos tienen un objeto brillante más grande (`area`) con mayor contraste respecto de su entorno (`obj_contrast`), una asimetría (`skewness`) positiva y una curtosis (`kurtosis`) mayor debido a la mancha brillante y, en la GLCM, un contraste menor y una homogeneidad mayor, porque una mancha suave reemplaza parte del fondo moteado. Estos son los números que la descripción de un radiólogo ("una densidad bien circunscrita") habría llevado a un ingeniero a calcular.

### 3.4 La línea base clásica

El bloque de código entrena los tres clasificadores de las Unidades 3–4 —5 vecinos más cercanos, una máquina de vectores de soporte con kernel RBF<sup class="c"><a href="#r37">[37]</a></sup> y un bosque aleatorio de 200 árboles<sup class="c"><a href="#r38">[38]</a></sup>— con las catorce características y, como comparación, con los 256 píxeles crudos, usando validación cruzada estratificada de 5 pliegues. Los pliegues quedan fijados por la semilla y son los mismos para todos los clasificadores, de modo que las diferencias entre filas son diferencias entre métodos y no entre particiones; la Sección 5 hace explícito este diseño pareado. La función auxiliar `sc_fit` estandariza con las filas de entrenamiento y aplica el mismo centro y la misma escala a las filas de prueba: el escalamiento dentro del pliegue que exigía la caja de advertencia de la Sección 2.1.



In [ ]:
flat_patch <- function(Xarr) t(apply(Xarr, 1, function(m) as.vector(t(m))))   # aplana cada parche, fila por fila
nod_P <- flat_patch(nod$X)                                                    # 1000 x 256
strat_folds <- function(y, k=5){ f <- integer(length(y))
  for (cl in unique(y)){ idx <- which(y==cl); f[idx] <- sample(rep(1:k, length.out=length(idx))) }; f }
sc_fit <- function(Xtr,ytr,Xte,model){ ce <- colMeans(Xtr); s <- apply(Xtr,2,sd); s[s==0] <- 1
  Ztr <- scale(Xtr,ce,s); Zte <- scale(Xte,ce,s)
  if (model=="knn") as.character(knn(Ztr,Zte,factor(ytr),k=5))
  else if (model=="svm") as.character(predict(svm(Ztr,factor(ytr)), Zte))
  else as.character(predict(randomForest(Ztr,factor(ytr),ntree=200), Zte)) }
cv_acc <- function(X,y,model,folds){ sapply(1:max(folds), function(f){
  te <- folds==f; mean(sc_fit(X[!te,],y[!te],X[te,],model)==as.character(y[te])) }) }
set.seed(0); folds <- strat_folds(nod$y, 5)
baseline <- sapply(c(`k-NN (k=5)`="knn", `SVM (RBF)`="svm", `random forest (200)`="rf"), function(m)
  c(`14 features`=sprintf("%.3f +/- %.3f", mean(cv_acc(nod_F,nod$y,m,folds)), sd(cv_acc(nod_F,nod$y,m,folds))),
    `256 pixels`=sprintf("%.3f +/- %.3f", mean(cv_acc(nod_P,nod$y,m,folds)), sd(cv_acc(nod_P,nod$y,m,folds)))))
cat("Nodule task, accuracy (mean +/- sd over 5 stratified folds):\n"); print(t(baseline), quote=FALSE)



Con las catorce características diseñadas, los tres clasificadores alcanzan exactitudes que van desde fines de los ochenta hasta comienzos de los noventa (en porcentaje), siendo el bosque aleatorio el mejor de ellos; con los píxeles crudos todos empeoran —el bosque en unos pocos puntos, la máquina de vectores de soporte en unos diez y k vecinos más cercanos en más de veinte—, porque la distancia euclidiana entre vectores de 256 píxeles está dominada por la textura de fondo y no por el nódulo. La lección es la misma con la que convivió el campo durante dos décadas: con mil imágenes, *las buenas características superan a los píxeles crudos*, y la calidad del sistema era la calidad de las características del ingeniero. Ten presente la primera columna: es la vara que el perceptrón debe superar.

<div class="box bridge"><h4>Puente hacia la radiómica y hacia BGM324</h4><p>El extractor anterior es una versión de juguete de la <strong>radiómica</strong>, la disciplina que calcula cientos de características estandarizadas de primer orden, de textura y de forma a partir de lesiones segmentadas en CT, MRI y PET, y las entrega a los clasificadores de las Unidades 3–4 para predecir la histología, el estado mutacional o la sobrevida. Sus definiciones de características descienden directamente del artículo de Haralick de 1973.<sup class="c"><a href="#r13">[13]</a></sup> El flujo de trabajo es idéntico al modelado QSAR del curso de bioinformática: un objeto (una lesión, una molécula) se describe mediante un vector de descriptores, y el modelo es tan bueno —y tan reproducible— como lo sean los descriptores.</p></div>

## 4. El perceptrón multicapa como clasificador de imágenes (8.1)

### 4.1 Aplanar, estandarizar, entrenar

La Unidad 6 construyó el perceptrón multicapa: una capa de entrada de \( p \) unidades, una capa oculta que calcula \( h = \phi(\mathbf{W}\mathbf{x} + \mathbf{b}) \) con una activación no lineal \( \phi \), y una capa de salida softmax, ajustado minimizando la función de pérdida de entropía cruzada. Nada en esa descripción menciona imágenes: para clasificar una imagen se la aplana (Sección 1.2), se estandariza cada posición de píxel sobre el conjunto de entrenamiento y el resultado se trata como un vector de características de longitud \( p = HW \). Para los glifos de 8 × 8, \( p = 64 \); con una sola capa oculta de \( m \) unidades, la red tiene \( 64m + m \) pesos hacia la capa oculta y \( 6m + 6 \) hacia la salida de seis clases. El perceptrón estándar de R es `nnet`,<sup class="c"><a href="#r35">[35]</a></sup> que ajusta una **única capa oculta** con activación logística mediante el optimizador BFGS; la Sección 5 usa `neuralnet` y un entrenador escrito desde cero cuando se necesita más de una capa o un optimizador distinto. El bloque de código evalúa el perceptrón con la metodología del Suplemento S1 —**validación cruzada estratificada de 5 pliegues repetida**, con dos repeticiones, lo que entrega diez estimaciones de la exactitud de prueba a partir de las cuales se reportan una media y una desviación estándar.



In [ ]:
make_glyphs <- function(per){ n <- per*length(GLYPHS); X <- matrix(0,n,SG*SG); y <- character(n); k <- 1
  for (cls in GLYPHS) for (i in 1:per){ X[k,] <- as.vector(draw_glyph(cls)); y[k] <- cls; k <- k+1 }
  list(X=X, y=factor(y, levels=GLYPHS)) }
set.seed(1); G <- make_glyphs(120)                      # 720 glifos, 120 por clase
Gctr <- colMeans(G$X); Gsd <- apply(G$X,2,sd); Gsd[Gsd==0] <- 1
Gs <- scale(G$X, Gctr, Gsd)                             # estandarizado una sola vez para los modelos de visualización
mlp_cv <- function(X, y, size, folds, decay=1e-3, maxit=200){      # predicciones fuera del pliegue, nnet multiclase
  pred <- factor(rep(NA, length(y)), levels=levels(y))
  for (f in 1:max(folds)){ te <- folds==f; ce <- colMeans(X[!te,]); s <- apply(X[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(X[!te,],ce,s), class.ind(y[!te]), size=size, softmax=TRUE, maxit=maxit,
                trace=FALSE, decay=decay, MaxNWts=1e5)
    pred[te] <- factor(predict(fit, scale(X[te,],ce,s), type="class"), levels=levels(y)) }
  pred }
set.seed(10)
scores <- unlist(lapply(1:2, function(rep){ fo <- strat_folds(G$y,5); oof <- mlp_cv(G$X,G$y,16,fo)
  sapply(1:5, function(f) mean(oof[fo==f]==G$y[fo==f])) }))
set.seed(0); fit_all <- nnet(Gs, class.ind(G$y), size=16, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
npar <- length(fit_all$wts)
cat(sprintf("architecture 64 -> 16 -> 6 : %d trainable parameters for %d images\n", npar, length(G$y)))
cat("accuracy per split:", round(scores,3), "\n")
cat(sprintf("repeated stratified 5-fold CV accuracy: %.3f +/- %.3f  (min %.3f, max %.3f)\n",
            mean(scores), sd(scores), min(scores), max(scores)))



El perceptrón clasifica correctamente la gran mayoría de los glifos reservados, con una desviación estándar entre las diez particiones de un par de puntos porcentuales. Observa *para qué* sirve la desviación estándar: una red que obtuviera un punto más en una sola partición no sería demostrablemente mejor que esta, porque diferencias de ese tamaño en una sola partición ocurren entre repeticiones del mismo modelo. Por eso la Sección 5 reporta cada comparación con su dispersión entre pliegues.

### 4.2 Qué clases se confunden

La exactitud resume; la matriz de confusión (Suplemento S1) explica. El bloque de código obtiene una predicción fuera del pliegue (out-of-fold) para cada glifo —cada uno predicho por un modelo que no se entrenó con él— y tabula la clase verdadera frente a la predicha.



In [ ]:
set.seed(2); foldsG <- strat_folds(G$y, 5)
y_oof <- mlp_cv(G$X, G$y, 16, foldsG)
C <- table(truth=G$y, predicted=y_oof)
cat(sprintf("out-of-fold accuracy: %.3f | errors: %d of %d\n",
            mean(y_oof==G$y), sum(y_oof!=G$y), length(G$y)))
off <- which(C>0 & row(C)!=col(C), arr.ind=TRUE)
ord <- order(-C[off]); top <- head(off[ord,,drop=FALSE], 5)
cat("most frequent confusions (true -> predicted : count): ",
    paste(sprintf("%s->%s: %d", GLYPHS[top[,1]], GLYPHS[top[,2]], C[top]), collapse=", "), "\n")
recall <- round(diag(C)/rowSums(C), 3); cat("per-class recall:", paste(GLYPHS, recall, sep="="), "\n")
par(mar=c(5,5,3,1))
image(1:6, 1:6, t(C[6:1,]), col=colorRampPalette(c("white", palette_neg))(32), axes=FALSE,
      xlab="predicted", ylab="true", main="Confusion matrix (out-of-fold, 5-fold CV)")
axis(1, 1:6, GLYPHS, las=2); axis(2, 1:6, rev(GLYPHS), las=1)
for (a in 1:6) for (b in 1:6) text(b, 7-a, C[a,b], cex=0.9, col=if (C[a,b]>max(C)/2) "white" else "black")



<p class="caption"><b>Figura 6.</b> La matriz de confusión del perceptrón sobre los 720 glifos. Los errores no se distribuyen de manera uniforme: el <code>ring</code> es la clase más confundible, porque a 8 × 8 píxeles y con mucho ruido un anillo ruidoso se parece a una <code>cross</code> o a una barra, y la <code>cross</code> comparte sus trazos vertical y horizontal con <code>vbar</code> y <code>hbar</code>. La sensibilidad (recall) por clase, impresa sobre la figura, va desde fines de los ochenta (en porcentaje) para el <code>ring</code> hasta cerca de uno para las barras, que se separan con nitidez. Un análogo radiológico sería una matriz de confusión de neumonía, edema y atelectasia, tres causas de opacidad que los propios radiólogos confunden; la matriz muestra dónde se necesita un segundo lector.</p>



In [ ]:
wrong <- which(y_oof != G$y)[1:min(20, sum(y_oof!=G$y))]
par(mfrow=c(2,10), mar=c(0.3,0.3,1.8,0.3), oma=c(0,0,2,0))
for (k in wrong){ image(t(matrix(G$X[k,],SG,SG)[SG:1,]), col=grey.colors(32,1,0), axes=FALSE, zlim=c(0,1))
  title(sprintf("%s -> %s", substr(as.character(G$y[k]),1,4), substr(as.character(y_oof[k]),1,4)), cex.main=0.75); box() }
mtext(sprintf("The first %d of the %d out-of-fold errors (true -> said)", length(wrong), sum(y_oof!=G$y)),
      outer=TRUE, line=0.3, cex=0.9, font=2)



<p class="caption"><b>Figura 7.</b> Veinte de los glifos mal clasificados, con la etiqueta verdadera y la respuesta de la red. Varios son ambiguos incluso para una persona con esta resolución y este nivel de ruido —un anillo tenue cortado por el ruido se lee como una cruz, una barra corta como una diagonal—, y ese es justamente el punto: con ocho píxeles por lado, a veces la información no está en la imagen, y ningún clasificador puede recuperarla. Mirar los errores es el diagnóstico más barato del campo, y es la etapa en la que se descubrieron, en conjuntos de datos reales, los marcadores de hospital de la Sección 7.</p>

### 4.3 Lo que ha aprendido la primera capa: los pesos como plantillas

Cada unidad oculta de la primera capa calcula \( \phi(\mathbf{w}_k^\top \mathbf{x} + b_k) \), una suma ponderada sobre los 64 píxeles. El vector de pesos \( \mathbf{w}_k \) tiene una entrada por píxel y, por lo tanto, puede reorganizarse como una imagen de 8 × 8 y mostrarse. Una unidad cuyos pesos son positivos donde se espera un trazo y negativos en el resto actúa como una **plantilla** —un filtro adaptado (matched filter) para una configuración de forma—, exactamente como las plantillas de medias por clase de la Unidad 1, salvo que ahora las plantillas se aprenden para ser *útiles para discriminar* en lugar de ser promedios, y la capa de salida combina 16 de ellas.



In [ ]:
# extrae los 64 pesos de entrada de cada unidad oculta del vector de pesos de nnet (disposición: por unidad oculta, primero el sesgo y luego las entradas)
p <- SG*SG; m <- 16
W1 <- sapply(1:m, function(k) fit_all$wts[((k-1)*(p+1)+2):(k*(p+1))])   # 64 x 24
ord <- order(-colSums(abs(W1)))
vmax <- max(abs(W1))
par(mfrow=c(2,8), mar=c(0.3,0.3,1.4,0.3), oma=c(0,0,2,0))
rb <- colorRampPalette(c(palette_neg,"white",palette_pos))(32)
for (k in ord){ image(t(matrix(W1[,k],SG,SG)[SG:1,]), col=rb, axes=FALSE, zlim=c(-vmax,vmax))
  title(sprintf("unit %d", k), cex.main=0.75); box() }
mtext("First-layer weights as 8 x 8 templates (blue negative, orange positive); all 16 units",
      outer=TRUE, line=0.3, cex=0.9, font=2)
cat(sprintf("weight matrix: %d x %d | weights range [%.2f, %.2f]\n", nrow(W1), ncol(W1), min(W1), max(W1)))



<p class="caption"><b>Figura 8.</b> Las 16 unidades de la primera capa, cada una mostrada como la imagen de 8 × 8 de sus pesos. Algunas son detectores de trazos reconocibles —una cresta vertical positiva (naranja) con flancos negativos (azules) responde a una barra vertical; una banda diagonal responde a <code>diagL</code> o <code>diagR</code>— y muchas son combinaciones ruidosas que solo resultan útiles en conjunto con las demás. Las plantillas son <em>globales</em>: cada unidad mira los 64 píxeles con sus propios pesos fijos, de modo que un detector de trazos que se activa ante una barra a la izquierda es una unidad distinta de otra que detecta la misma barra a la derecha. La Sección 6 muestra que esto es exactamente lo que hace al perceptrón frágil ante los desplazamientos y por qué las redes convolucionales comparten pesos entre posiciones.</p>

### 4.4 El perceptrón con los nódulos simulados

El bloque de código repite la comparación de la Sección 3.4 agregando el perceptrón: una vez con los 256 píxeles crudos y otra con las 14 características diseñadas. Para el modelo de píxeles se usa una penalización L2 fuerte (`decay = 0.5`), porque con 256 entradas y 1,000 imágenes la red sobreajusta severamente con una penalización pequeña; esta elección es en sí misma una decisión de topología del tipo que la Sección 5 estudia de forma sistemática. `nnet` se entrena para un resultado binario con `entropy = TRUE` (pérdida logística).



In [ ]:
mlp_bin_cv <- function(X, y, size, folds, decay){ p <- numeric(length(y))
  for (f in 1:max(folds)){ te <- folds==f; ce <- colMeans(X[!te,]); s <- apply(X[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(X[!te,],ce,s), y[!te], size=size, entropy=TRUE, maxit=250, trace=FALSE, decay=decay, MaxNWts=1e5)
    p[te] <- as.vector(predict(fit, scale(X[te,],ce,s))) }; p }
set.seed(0); pF <- mlp_bin_cv(nod_F,nod$y,6,folds,0.5); pP <- mlp_bin_cv(nod_P,nod$y,6,folds,0.5)
accF <- sapply(1:5, function(f) mean((pF[folds==f]>0.5)==nod$y[folds==f]))
accP <- sapply(1:5, function(f) mean((pP[folds==f]>0.5)==nod$y[folds==f]))
mlp_rows <- rbind(`MLP (6), decay=0.5` = c(`14 features`=sprintf("%.3f +/- %.3f",mean(accF),sd(accF)),
                                           `256 pixels`=sprintf("%.3f +/- %.3f",mean(accP),sd(accP))))
cat("Nodule task, accuracy (mean +/- sd over the same 5 stratified folds as Section 3.4):\n")
print(rbind(t(baseline), mlp_rows), quote=FALSE)



Con las características diseñadas, el perceptrón se ubica entre los clasificadores clásicos: las características contienen la información, y cualquier clasificador razonable encuentra una frontera. Con los píxeles crudos, el perceptrón con la penalización fuerte alcanza una exactitud de entre el ochenta y el ochenta y cinco por ciento, aproximadamente, mucho mejor que k vecinos más cercanos con píxeles y comparable a la máquina de vectores de soporte, pero igual o inferior al bosque aleatorio con píxeles y **por debajo de los modelos basados en características**. Este es, en miniatura, el estado del arte de alrededor de 2010: con mil imágenes de entrenamiento, una red poco profunda con píxeles no supera a unas buenas características. La ventaja de las características diseñadas es una ventaja de *conocimiento previo*: alguien sabía que un nódulo es un objeto brillante, redondo y con contraste local, y lo escribió como código. La Sección 6 muestra cómo dar a una red un conocimiento previo comparable sobre las imágenes sin diseñar a mano las características.

### 4.5 Probabilidades y umbrales en un contexto de tamizaje

Un perceptrón entrega una probabilidad \( \hat{p}(\text{nodule} \mid \mathbf{x}) \); la etiqueta se asigna cuando \( \hat{p} \) supera un umbral. El valor por defecto de 0.5 es adecuado cuando ambos errores cuestan lo mismo y la prevalencia de entrenamiento coincide con la del despliegue; ninguna de las dos condiciones se cumple en el tamizaje, donde un cáncer no detectado cuesta más que una nueva citación (recall) y la prevalencia es de unos pocos por ciento. El bloque de código calcula la sensibilidad, la especificidad y la curva ROC<sup class="c"><a href="#r39">[39]</a></sup> a partir de las probabilidades fuera del pliegue y las convierte en valores predictivos positivos a dos prevalencias con el teorema de Bayes (Unidad 3): \( \mathrm{PPV} = \frac{\text{se}\,\pi}{\text{se}\,\pi + (1-\text{sp})(1-\pi)} \).



In [ ]:
p_oof <- pP                                 # reutiliza las probabilidades fuera del pliegue del MLP de píxeles de la Sección 4.4
roc_obj <- roc(nod$y, p_oof, quiet=TRUE, direction="<")
ppv <- function(se,sp,pi) se*pi/(se*pi + (1-sp)*(1-pi))
ops <- do.call(rbind, lapply(c(0.1,0.3,0.5,0.7,0.9), function(t){ pred <- p_oof>=t
  se <- sum(pred & nod$y==1)/sum(nod$y==1); sp <- sum(!pred & nod$y==0)/sum(nod$y==0)
  data.frame(threshold=t, sensitivity=round(se,3), specificity=round(sp,3),
             `PPV_prev50`=round(ppv(se,sp,0.5),3), `PPV_prev2`=round(ppv(se,sp,0.02),3)) }))
cat(sprintf("ROC-AUC of the out-of-fold probabilities: %.3f\n", as.numeric(auc(roc_obj))))
plot(1-ops$specificity, ops$sensitivity, type="n", xlim=c(0,1), ylim=c(0,1),
     xlab="1 - specificity", ylab="sensitivity", main="ROC curve of the pixel perceptron")
lines(1-roc_obj$specificities, roc_obj$sensitivities, col=palette_neg, lwd=2)
abline(0,1,lty=2, col="grey"); points(1-ops$specificity, ops$sensitivity, pch=19, col=palette_pos)
text(1-ops$specificity, ops$sensitivity, paste0("t=",ops$threshold), pos=4, cex=0.7)
legend("bottomright", sprintf("pixel MLP, AUC = %.3f", as.numeric(auc(roc_obj))), lwd=2, col=palette_neg, bty="n", cex=0.8)
print(ops, row.names=FALSE)



<p class="caption"><b>Figura 9.</b> La curva ROC del perceptrón de píxeles en la tarea de nódulos, construida a partir de probabilidades fuera del pliegue, con cinco umbrales marcados. Bajar el umbral de 0.5 a 0.1 desplaza el punto de operación hacia arriba y hacia la derecha: la sensibilidad sube y la especificidad baja, como muestra la tabla. Las dos últimas columnas son la consecuencia de la prevalencia que se derivó en la Unidad 3: con la prevalencia del 50 % de este conjunto de datos balanceado, el valor predictivo positivo es alto en todos los umbrales, pero con una prevalencia de tamizaje del 2 % la mayoría de los positivos son falsos en cualquier umbral, porque la tasa de falsos positivos se aplica a la gran mayoría sana. La curva y su AUC son propiedades del modelo; el PPV es una propiedad del modelo <em>y</em> de la población, y una herramienta de tamizaje debe evaluarse con la prevalencia en la que se usará.</p>

<div class="exercise"><h4>Ejercicio 2</h4><p>Elige el umbral que da una sensibilidad de al menos 0.95 (léelo de <code>roc_obj</code> con <code>coords(roc_obj, "all")</code>) y calcula el número de falsos positivos por cada 1,000 pacientes tamizados con una prevalencia del 2 %. Luego repite con el bosque aleatorio basado en características de la Sección 3.4 (usa su salida de probabilidades por clase, <code>predict(..., type="prob")</code>). ¿Qué modelo desplegarías como <em>segundo lector</em>, cuyos positivos revisa un radiólogo, y cuál como herramienta de <em>triaje</em> que descarta los negativos sin que nadie los lea?</p></div>

## 5. Pruebas experimentales de la topología de la red (8.2)

### 5.1 Cómo realizar un experimento honesto

La topología de un perceptrón —capas ocultas, unidades por capa, activación, intensidad de la regularización, optimizador, tasa de aprendizaje, regla de detención— no se aprende de los datos; la elige el ingeniero, y el programa del curso te pide elegirla *experimentalmente*. Cuatro reglas, todas del Suplemento S1, hacen que un experimento así sea confiable:

1. **Las mismas particiones para todas las configuraciones** (fijadas con `set.seed`), de modo que las diferencias entre configuraciones no se confundan con diferencias entre particiones: la comparación es *pareada*.
2. **Repite, y reporta la varianza.** Una diferencia menor que las desviaciones estándar entre particiones no es un hallazgo.
3. **Cambia un factor a la vez**, o cruza los factores en un diseño factorial, como en la grilla profundidad × ancho de más abajo.
4. **No reportes el mejor puntaje de validación cruzada como el rendimiento esperado.** Elegir la mejor de muchas configuraciones con los mismos pliegues ajusta esos pliegues, y el puntaje elegido está sesgado de forma optimista;<sup class="c"><a href="#r15">[15]</a></sup> la Sección 5.6 muestra el remedio.

<div class="box warn"><h4>Un detalle de R que da forma a esta sección</h4><p>El paquete <code>nnet</code> de R ajusta <em>una</em> capa oculta con activación logística, optimizada con BFGS (un método cuasi-Newton de lote completo). No tiene noción de profundidad más allá de una capa, ni de activación lineal rectificada, ni de tasa de aprendizaje y minilotes. Para variar esos factores con fidelidad, esta sección usa tres herramientas: <code>nnet</code> para el experimento de <strong>ancho</strong> (Sección 5.2) y el de <strong>regularización</strong> (su argumento <code>decay</code> es la penalización L2); <code>neuralnet</code><sup class="c"><a href="#r36">[36]</a></sup> para el experimento de <strong>profundidad</strong>, ya que admite varias capas ocultas y permite elegir entre activación logística y <code>tanh</code>; y un entrenador <strong>escrito desde cero</strong> de descenso de gradiente por minilotes (Sección 5.5) para el experimento de tasa de aprendizaje, momento y tamaño de lote, que además nos permite usar la unidad lineal rectificada de la que <code>nnet</code> carece. La edición en Python usaba un único <code>MLPClassifier</code> para todo esto; la edición en R lo reparte entre las tres herramientas, cuya unión cubre el mismo terreno.</p></div>

### 5.2 Profundidad × ancho

El primer experimento cruza el número de capas ocultas (0, 1, 2, 3) con el número de unidades por capa (8, 16, 24). Una red con cero capas ocultas es una regresión logística multinomial: la línea base lineal. Para que la optimización multicapa sea rápida y converja de manera confiable, la grilla de profundidad entrena `neuralnet` sobre un PCA de 12 componentes de los píxeles de los glifos (una representación no supervisada fijada de antemano, por lo que no introduce fuga de etiquetas); el `nnet` de una sola capa con píxeles crudos de la Sección 4.1 alcanzó una exactitud comparable, lo que vincula ambas representaciones. El bloque de código ajusta la grilla con validación cruzada de 3 pliegues y la dibuja como un mapa de calor.



In [ ]:
pc <- prcomp(G$X, center=TRUE, scale.=FALSE); Z <- scale(pc$x[,1:12]); colnames(Z) <- paste0("z",1:12)
Yoh <- class.ind(G$y); colnames(Yoh) <- GLYPHS
dz <- data.frame(Z, Yoh, check.names=FALSE)
form <- as.formula(paste(paste(GLYPHS, collapse="+"), "~", paste0("z",1:12, collapse="+")))
nn_cv <- function(hidden, folds){ sapply(1:max(folds), function(f){ te <- folds==f
  if (is.null(hidden)){ m <- multinom(cls ~ ., data=data.frame(Z, cls=G$y)[!te,], trace=FALSE)
    pr <- predict(m, data.frame(Z)[te,]) }
  else { nn <- try(neuralnet(form, dz[!te,], hidden=hidden, linear.output=FALSE, act.fct="logistic",
                             stepmax=1e5, threshold=0.1), silent=TRUE)
    if (inherits(nn,"try-error") || is.null(nn$weights)) return(NA)
    pr <- factor(GLYPHS[max.col(predict(nn, dz[te,]))], levels=GLYPHS) }
  mean(pr==G$y[te]) }) }
set.seed(3); fo <- strat_folds(G$y, 3); widths <- c(8,16,24)
M <- matrix(NA, 4, 3, dimnames=list(c("0 (linear)","1 layer","2 layers","3 layers"), paste0(widths," u")))
Sg <- M
lin <- nn_cv(NULL, fo); M[1,] <- mean(lin); Sg[1,] <- sd(lin)
for (j in seq_along(widths)){ w <- widths[j]
  for (d in 1:3){ a <- nn_cv(rep(w, d), fo); M[d+1,j] <- mean(a, na.rm=TRUE); Sg[d+1,j] <- sd(a, na.rm=TRUE) } }
best <- which(M==max(M,na.rm=TRUE), arr.ind=TRUE)[1,]
cat(sprintf("best cell: %s, %s -> %.3f | linear baseline %.3f\n",
            rownames(M)[best[1]], colnames(M)[best[2]], max(M,na.rm=TRUE), M[1,1]))
print(round(M,3))
par(mar=c(4,7,2,1)); image(1:3, 1:4, t(M[4:1,]), col=colorRampPalette(c("white",palette_acc))(32),
      axes=FALSE, xlab="units per layer", ylab=""); box()
axis(1, 1:3, paste0(widths)); axis(2, 1:4, rev(rownames(M)), las=1)
for (a in 1:4) for (b in 1:3) text(b, 5-a, sprintf("%.2f\n+/-%.2f", M[a,b], Sg[a,b]), cex=0.75)
title("Glyphs: CV accuracy by depth and width", cex.main=0.95)



<p class="caption"><b>Figura 10.</b> La grilla profundidad × ancho en los glifos (el modelo lineal no tiene ancho y se repite a lo largo de su fila). Tres observaciones. Primero, el salto grande y confiable es el que va del modelo lineal a una sola capa oculta: una capa oculta eleva la exactitud muy por encima de la línea base lineal, porque las clases no son linealmente separables en el espacio de las componentes principales. Segundo, agregar más capas produce aquí solo un pequeño aumento adicional —la red de tres capas más ancha es la mejor celda—, pero los incrementos de una capa a dos o tres son comparables a las desviaciones estándar entre pliegues impresas en cada celda, de modo que la grilla no ordena de forma robusta las configuraciones multicapa: con esta muestra son, dentro del ruido, equivalentes. Tercero, ese es exactamente el comportamiento esperado con unos pocos cientos de imágenes: la profundidad adicional agrega una capacidad que los datos apenas pueden restringir, de modo que cualquier ganancia es pequeña e incierta. "Más grande es mejor" es un eslogan sobre redes entrenadas con millones de imágenes; con una muestra pequeña, el reporte honesto es la grilla completa con sus barras de error, no la mejor celda, cuyo valor está sesgado hacia arriba precisamente por ser el máximo (Sección 5.6).</p>

### 5.3 Curvas de aprendizaje: cómo depende el rendimiento de la muestra

Una curva de aprendizaje grafica la exactitud de validación cruzada frente al número de imágenes de entrenamiento. Responde la pregunta que la grilla no puede responder: *¿ayudarían más datos, y qué modelo se beneficiaría?* El bloque de código dibuja las curvas de un `nnet` de una sola capa angosto y de uno ancho, junto con la exactitud de entrenamiento.



In [ ]:
lc <- function(size, fracs=c(0.15,0.35,0.6,1.0)){ set.seed(5); fo <- strat_folds(G$y,3)
  t(sapply(fracs, function(fr){ tr_acc <- cv_acc <- numeric(3)
    for (f in 1:3){ te <- fo==f; tri <- sample(which(!te)); tri <- tri[1:round(fr*length(tri))]
      ce <- colMeans(G$X[tri,]); s <- apply(G$X[tri,],2,sd); s[s==0] <- 1
      fit <- nnet(scale(G$X[tri,],ce,s), class.ind(G$y[tri]), size=size, softmax=TRUE, maxit=200,
                  trace=FALSE, decay=1e-3, MaxNWts=1e5)
      tr_acc[f] <- mean(predict(fit, scale(G$X[tri,],ce,s), type="class")==G$y[tri])
      cv_acc[f] <- mean(predict(fit, scale(G$X[te,],ce,s), type="class")==G$y[te]) }
    c(n=round(fr*length(G$y)*2/3), train=mean(tr_acc), cv=mean(cv_acc), sd=sd(cv_acc)) })) }
L8 <- lc(6); L128 <- lc(32)
plot(L8[,"n"], L8[,"cv"], type="b", pch=19, col=palette_pos, ylim=c(0.4,1.02),
     xlab="number of training images", ylab="accuracy", main="Learning curves on the glyphs")
lines(L8[,"n"], L8[,"train"], lty=2, col=palette_pos); lines(L128[,"n"], L128[,"cv"], type="b", pch=19, col=palette_neg)
lines(L128[,"n"], L128[,"train"], lty=2, col=palette_neg)
legend("bottomright", c("6 units, CV","6 units, train","32 units, CV","32 units, train"),
       col=c(palette_pos,palette_pos,palette_neg,palette_neg), lty=c(1,2,1,2), pch=c(19,NA,19,NA), bty="n", cex=0.75)
cat("CV accuracy by training size:\n"); print(round(rbind(`6 units`=L8[,"cv"], `32 units`=L128[,"cv"]),3))



<p class="caption"><b>Figura 11.</b> Curvas de aprendizaje de los perceptrones de 6 unidades (naranja) y de 32 unidades (azul). La red ancha va adelante por unos pocos puntos con todos los tamaños de muestra y sigue mejorando con el conjunto de entrenamiento completo, mientras que su exactitud de entrenamiento (línea discontinua) se mantiene alta; así, la brecha entre las líneas azules discontinua y continua —el sobreajuste— solo se cierra a medida que crece la muestra. La red angosta sube más lentamente y se estanca varios puntos más abajo, porque está limitada por su propia capacidad (subajuste), y más datos la ayudarían menos. Estas curvas son el diagnóstico estándar para decidir si conviene reunir más imágenes —lo que en imágenes médicas cuesta meses y aprobaciones éticas— o cambiar el modelo.</p>

### 5.4 Activación y regularización

Dos factores más. La **penalización L2** (`decay` en `nnet`) contrae los pesos hacia cero, el equivalente en el perceptrón de la penalización ridge de la Unidad 2; se varía en cinco órdenes de magnitud en la red de 32 unidades. La **activación** es la sigmoide logística o la tangente hiperbólica; `neuralnet` ofrece ambas, y el bloque de código las compara sobre la representación de componentes principales. (La unidad lineal rectificada, que ni `nnet` ni `neuralnet` ofrecen como función diferenciable incorporada, se usa en el entrenador escrito desde cero de la Sección 5.5, donde su efecto sobre la velocidad de entrenamiento es visible.)



In [ ]:
set.seed(6); fo <- strat_folds(G$y, 3); decays <- c(1e-3, 1e-1, 1, 10, 100)
vc <- sapply(decays, function(al){ tr <- cv <- numeric(3)
  for (f in 1:3){ te <- fo==f; ce <- colMeans(G$X[!te,]); s <- apply(G$X[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(G$X[!te,],ce,s), class.ind(G$y[!te]), size=24, softmax=TRUE, maxit=200,
                trace=FALSE, decay=al, MaxNWts=1e5)
    tr[f] <- mean(predict(fit, scale(G$X[!te,],ce,s), type="class")==G$y[!te])
    cv[f] <- mean(predict(fit, scale(G$X[te,],ce,s), type="class")==G$y[te]) }
  c(train=mean(tr), cv=mean(cv), sd=sd(cv)) })
act <- sapply(c("logistic","tanh"), function(a){ acc <- numeric(3)
  for (f in 1:3){ te <- fo==f
    nn <- try(neuralnet(form, dz[!te,], hidden=16, linear.output=FALSE, act.fct=a, stepmax=1e5, threshold=0.1), silent=TRUE)
    acc[f] <- if (inherits(nn,"try-error")||is.null(nn$weights)) NA else
      mean(factor(GLYPHS[max.col(predict(nn, dz[te,]))], levels=GLYPHS)==G$y[te]) }
  mean(acc, na.rm=TRUE) })
plot(decays, vc["cv",], log="x", type="b", pch=19, col=palette_neg, ylim=c(min(vc)-0.02,1.0),
     xlab="decay (L2 penalty)", ylab="accuracy", main="Validation curve over the L2 penalty")
lines(decays, vc["train",], type="b", pch=0, lty=2, col="grey40")
arrows(decays, vc["cv",]-vc["sd",], decays, vc["cv",]+vc["sd",], angle=90, code=3, length=0.03, col=palette_neg)
legend("bottomleft", c("CV accuracy +/- sd","training accuracy"), col=c(palette_neg,"grey40"), lty=c(1,2), pch=c(19,0), bty="n", cex=0.8)
cat("CV accuracy by decay:", paste(decays, round(vc["cv",],3), sep="="), "\n")
cat(sprintf("activation (neuralnet, 16 units): logistic %.3f | tanh %.3f\n", act["logistic"], act["tanh"]))



<p class="caption"><b>Figura 12.</b> Exactitud de entrenamiento (gris) y de validación cruzada (azul) a medida que crece la penalización L2. La curva es aproximadamente plana con penalizaciones pequeñas —en estos glifos a la red no le falta regularización— y luego cae con los valores más grandes, donde la penalización contrae hacia cero las plantillas de la Figura 8 y la exactitud de entrenamiento baja con ellas. La comparación de activaciones impresa muestra que las unidades logísticas y <code>tanh</code> alcanzan aquí una exactitud similar; con datos más grandes y más ruidosos el orden puede cambiar, y por eso estos son experimentos y no reglas.</p>

### 5.5 Optimizador, tasa de aprendizaje y tamaño de lote

Los pesos se ajustan por descenso de gradiente (Unidades 2 y 6): un **minilote** (mini-batch) de imágenes pasa por la red, el gradiente de la función de pérdida se calcula por retropropagación y los pesos se mueven en sentido contrario con un paso proporcional a la **tasa de aprendizaje**; el **momento** (momentum) agrega una fracción del paso anterior, lo que acelera el descenso a lo largo de direcciones consistentes. Como `nnet` oculta todo esto dentro de BFGS, el bloque de código implementa **desde cero**, en una docena de líneas de código matricial, una red softmax compacta de una capa oculta con activación lineal rectificada, y la entrena con varias configuraciones sobre una única partición 70/30, graficando la pérdida de entrenamiento por época: la **curva de pérdida**, lo primero que hay que mirar cuando el entrenamiento se comporta mal.



In [ ]:
relu <- function(x) pmax(x,0)
softmax_rows <- function(Z){ Z <- Z - apply(Z,1,max); E <- exp(Z); E/rowSums(E) }
train_mlp <- function(X, Y, hidden=16, lr=0.1, batch=32, epochs=25, momentum=0){
  n <- nrow(X); p <- ncol(X); K <- ncol(Y)
  W1 <- matrix(rnorm(p*hidden,0,sqrt(2/p)),p,hidden); b1 <- rep(0,hidden)
  W2 <- matrix(rnorm(hidden*K,0,sqrt(2/hidden)),hidden,K); b2 <- rep(0,K)
  vW1<-0*W1; vb1<-0*b1; vW2<-0*W2; vb2<-0*b2; loss <- numeric(epochs)
  for (ep in 1:epochs){ idx <- sample(n); el <- 0
    for (s in seq(1,n,by=batch)){ bi <- idx[s:min(s+batch-1,n)]; xb <- X[bi,,drop=FALSE]; yb <- Y[bi,,drop=FALSE]; mb <- nrow(xb)
      H <- relu(sweep(xb%*%W1,2,b1,"+")); O <- softmax_rows(sweep(H%*%W2,2,b2,"+")); el <- el - sum(yb*log(O+1e-9))
      dO <- (O-yb)/mb; dW2 <- t(H)%*%dO; db2 <- colSums(dO); dH <- (dO%*%t(W2))*(H>0); dW1 <- t(xb)%*%dH; db1 <- colSums(dH)
      vW1<-momentum*vW1-lr*dW1; W1<-W1+vW1; vb1<-momentum*vb1-lr*db1; b1<-b1+vb1
      vW2<-momentum*vW2-lr*dW2; W2<-W2+vW2; vb2<-momentum*vb2-lr*db2; b2<-b2+vb2 }
    loss[ep] <- el/n }
  list(loss=loss, acc=function(Xn,yn) mean(GLYPHS[max.col(softmax_rows(sweep(relu(sweep(Xn%*%W1,2,b1,"+"))%*%W2,2,b2,"+")))]==yn)) }
set.seed(2); tr <- sample(nrow(Gs), round(0.7*nrow(Gs))); Yg <- class.ind(G$y)
par(mfrow=c(1,2), mar=c(4,4,2,1))
for (mom in c(0, 0.9)){ plot(NA, xlim=c(1,25), ylim=c(1e-3,3), log="y", xlab="epoch", ylab="training loss",
    main=sprintf("%s", ifelse(mom==0,"plain SGD","SGD + momentum 0.9"))); cols <- c(palette_pos,palette_neg,palette_acc); i <- 1
  for (lr in c(0.01,0.1,0.5)){ set.seed(1); m <- train_mlp(Gs[tr,], Yg[tr,], lr=lr, batch=32, epochs=25, momentum=mom)
    lines(m$loss, col=cols[i], lwd=1.3); legend_txt <- sprintf("lr %.2f: test acc %.2f", lr, m$acc(Gs[-tr,],G$y[-tr])) 
    text(25, m$loss[25], labels=legend_txt, pos=2, cex=0.6, col=cols[i]); i <- i+1 } }



<p class="caption"><b>Figura 13.</b> Pérdida de entrenamiento por época para el descenso de gradiente por minilotes simple (izquierda) y para el mismo con momento 0.9 (derecha), con tres tasas de aprendizaje, y la exactitud de prueba tras 25 épocas anotada en cada curva. Con el descenso simple, la tasa de aprendizaje más pequeña apenas ha comenzado a converger; la tasa intermedia se comporta bien; la más grande converge más rápido, pero de manera menos suave. El momento acelera de forma drástica las tasas de aprendizaje pequeñas —la curva de 0.01 ahora llega a una pérdida baja—, pero con la tasa más grande se pasa de largo y la pérdida diverge, la firma característica de un paso demasiado largo. Cómo se <em>entrena</em> una red importa tanto como la <em>forma</em> que tiene, y un artículo que reporta una arquitectura sin la configuración de su optimizador no ha descrito un experimento reproducible.</p>

### 5.6 Validación cruzada anidada: elegir la topología sin favorecerla

Si la mejor celda de la grilla de la Sección 5.2 se reporta como "la exactitud de nuestro perceptrón", el número está sesgado hacia arriba: se seleccionó *porque* era la más alta entre varias estimaciones ruidosas en esos mismos pliegues. Varma y Simon midieron este sesgo en conjuntos de datos biomédicos pequeños y encontraron que puede superar las diferencias reales entre métodos.<sup class="c"><a href="#r15">[15]</a></sup> El remedio es la **validación cruzada anidada**: un ciclo *externo* reserva un pliegue de prueba; dentro de cada pliegue de entrenamiento, una validación cruzada *interna* elige la topología (aquí, el ancho de una única capa oculta); el modelo elegido se vuelve a ajustar con todo el pliegue de entrenamiento y se evalúa una sola vez en el pliegue de prueba externo. Los puntajes externos estiman el rendimiento del *procedimiento* "elegir el ancho por validación cruzada y entrenar", que es lo que efectivamente se desplegará.



In [ ]:
cand <- c(6, 16, 24)
inner_choose <- function(Xtr, ytr){ set.seed(1); ifo <- strat_folds(ytr, 3)
  acc <- sapply(cand, function(w) mean(sapply(1:3, function(f){ te <- ifo==f
    ce <- colMeans(Xtr[!te,]); s <- apply(Xtr[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(Xtr[!te,],ce,s), class.ind(ytr[!te]), size=w, softmax=TRUE, maxit=150, trace=FALSE, decay=1e-3, MaxNWts=1e5)
    mean(predict(fit, scale(Xtr[te,],ce,s), type="class")==ytr[te]) })))
  cand[which.max(acc)] }
set.seed(7); ofo <- strat_folds(G$y, 3); outer_acc <- numeric(3); chosen <- integer(3)
for (f in 1:3){ te <- ofo==f; w <- inner_choose(G$X[!te,], G$y[!te]); chosen[f] <- w
  ce <- colMeans(G$X[!te,]); s <- apply(G$X[!te,],2,sd); s[s==0] <- 1
  fit <- nnet(scale(G$X[!te,],ce,s), class.ind(G$y[!te]), size=w, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
  outer_acc[f] <- mean(predict(fit, scale(G$X[te,],ce,s), type="class")==G$y[te]) }
cat(sprintf("nested CV accuracy of 'choose width by inner CV, then train': %.3f +/- %.3f\n", mean(outer_acc), sd(outer_acc)))
cat("width chosen in each outer fold:", chosen, "\n")
cat(sprintf("for comparison, the best single-layer cell of the grid (selected on the same data): %.3f\n", max(M[2,], na.rm=TRUE)))



Los anchos elegidos por el ciclo interno se imprimen para cada pliegue externo, y la estimación anidada es cercana a la mejor celda de una sola capa de la grilla y, por lo general, ligeramente inferior. La brecha es pequeña aquí porque la grilla era pequeña y la señal fuerte; en un conjunto de datos de unos pocos cientos de pacientes con una grilla de cincuenta configuraciones, la brecha es habitualmente de varios puntos porcentuales, y es la diferencia entre una exactitud publicada y la exactitud que observará un hospital. El principio es el del Suplemento S1 llevado un nivel más arriba: todo lo que se *elige* usando datos debe elegirse dentro del ciclo que *evalúa*.

<div class="box key"><h4>Lo que establecieron los experimentos de topología</h4>
<ul>
<li>Con unos pocos cientos de imágenes pequeñas, la ganancia decisiva es la que va del modelo lineal a una sola capa oculta; agregar más capas cambia la exactitud en no más que el ruido entre pliegues, de modo que con este tamaño de muestra no se puede demostrar que la profundidad ayude.</li>
<li>Toda comparación necesita las mismas particiones, repeticiones y una desviación estándar; la mayoría de las diferencias de esta sección están dentro de una desviación estándar entre sí.</li>
<li>El optimizador, la tasa de aprendizaje, el momento y la penalización cambian el resultado tanto como la arquitectura, y deben reportarse junto con ella.</li>
<li>La configuración se elige con validación cruzada interna y el rendimiento se estima con validación cruzada externa; la mejor celda de una grilla no es una estimación de nada.</li>
</ul></div>

<div class="exercise"><h4>Ejercicio 3</h4><p>Agrega un ancho de 48 a la grilla de la Sección 5.2 (solo con una y dos capas ocultas, para ahorrar tiempo) y al conjunto de candidatos de la validación cruzada anidada. ¿Mejora la red más ancha la estimación anidada? Luego reduce el conjunto de datos a 40 glifos por clase (<code>make_glyphs(40)</code>) y vuelve a ejecutar la fila de una capa oculta de la grilla. ¿Qué ancho gana ahora, y qué tan grandes son las desviaciones estándar?</p></div>

## 6. Invariancia a la traslación, convolución, y por qué el perceptrón tiene la forma equivocada para las imágenes

### 6.1 Desplazar un glifo un píxel

Una unidad de la primera capa de un perceptrón multiplica cada píxel por su propio peso (Figura 8). Si la imagen se desplaza un píxel, cada píxel se encuentra ahora con el peso destinado a su vecino, y la respuesta de la unidad cambia por completo, aunque la imagen sea "la misma" para cualquier observador. La propiedad que debería tener un clasificador de imágenes —que la clase no cambie cuando el objeto se mueve— se llama **invariancia a la traslación**, y un perceptrón no tiene nada de ella, salvo lo que aprende, posición por posición, de la muestra de entrenamiento. El bloque de código mide el efecto: se entrena un perceptrón con los glifos en sus posiciones habituales y se evalúa con un conjunto nuevo de glifos desplazados uno, dos y tres píxeles.



In [ ]:
shift_flat <- function(X, dx, dy) t(apply(X, 1, function(v) as.vector(shift_img(matrix(v,SG,SG), dx, dy))))
set.seed(20); Gte <- make_glyphs(40)                       # glifos reservados nuevos
raw_mlp <- nnet(Gs, class.ind(G$y), size=16, softmax=TRUE, maxit=250, trace=FALSE, decay=1e-3, MaxNWts=1e5)
shifts <- list(c(0,0),c(1,0),c(0,1),c(1,1),c(2,0),c(2,2),c(3,0))
acc_raw <- sapply(shifts, function(s) mean(predict(raw_mlp, scale(shift_flat(Gte$X,s[1],s[2]),Gctr,Gsd), type="class")==Gte$y))
cat("test accuracy when the glyphs are shifted by (dx,dy):\n")
cat(paste(sapply(seq_along(shifts), function(i) sprintf("(%d,%d):%.3f", shifts[[i]][1], shifts[[i]][2], acc_raw[i])), collapse="  "), "\n")
k <- which(predict(raw_mlp, Gs, type="class")==G$y & G$y=="cross")[1]
par(mfrow=c(1,6), mar=c(1,1,2.4,1))
for (i in 1:5){ s <- shifts[[i]]; im <- shift_img(matrix(G$X[k,],SG,SG), s[1], s[2])
  pr <- predict(raw_mlp, scale(matrix(as.vector(im),1), Gctr, Gsd), type="class")
  image(t(im[SG:1,]), col=grey.colors(32,1,0), axes=FALSE); title(sprintf("shift (%d,%d)\nsaid %s", s[1],s[2],substr(pr,1,4)), cex.main=0.8); box() }
barplot(acc_raw, names.arg=sapply(shifts,function(s)paste0("(",s[1],",",s[2],")")), col=palette_neg,
        ylim=c(0,1), las=2, cex.names=0.6, main="accuracy vs shift", cex.main=0.8)



<p class="caption"><b>Figura 14.</b> Izquierda: un glifo reservado desplazado hasta dos píxeles, con la predicción del perceptrón bajo cada uno. Derecha: la exactitud de prueba en función del desplazamiento. La exactitud sin desplazamiento es alta; un desplazamiento de <em>un píxel</em> —un octavo de la imagen— ya la reduce, y dos o tres píxeles la llevan hacia el nivel del azar. Nada de los glifos cambió; solo cambió su posición. Una radiografía nunca se posiciona con precisión de un píxel, y una lesión puede aparecer en cualquier parte del campo de visión, de modo que esta fragilidad sería descalificante en la práctica. La Sección 2.3 ofrece un remedio, el aumento de datos con copias desplazadas; la Sección 6.4 mide hasta dónde llega y presenta el segundo remedio, que es arquitectónico.</p>

### 6.2 La convolución, a mano

Una **convolución** desliza una pequeña matriz de pesos —el **kernel** o **filtro**, típicamente de 3 × 3— sobre la imagen y, en cada posición, entrega la suma de los productos de los elementos del kernel por los píxeles que quedan bajo ellos:

<div class="eq">\[ (\mathbf{I} \star \mathbf{K})_{r,c} = \sum_{u=1}^{k} \sum_{v=1}^{k} K_{u,v}\, I_{r+u-1,\, c+v-1} \]
<span class="where">La salida es un <strong>mapa de características</strong> (feature map): grande donde la imagen se parece localmente al kernel. En el borde de la imagen, los vecinos faltantes se rellenan con ceros (relleno o padding, salida del mismo tamaño, "same") o se descartan ("valid", salida más pequeña). En sentido estricto, esta operación es una <em>correlación cruzada</em>; una convolución matemática primero invierte el kernel. Las bibliotecas de aprendizaje profundo calculan la correlación cruzada y la llaman convolución, y como los kernels se aprenden, la distinción es irrelevante. Sin relleno, un kernel de \( k \times k \) sobre una imagen de \( H \times W \) produce un mapa de \( (H-k+1) \times (W-k+1) \).</span></div>

Los mismos nueve pesos se aplican en cada posición: el kernel es una *única* plantilla compartida por toda la imagen, a diferencia de los 64 pesos independientes por unidad del perceptrón. Ese compartir pesos es lo que da a la operación su **equivariancia a la traslación** —si se desplaza la imagen, el mapa de características se desplaza con ella y, por lo demás, no cambia—, y el paso de pooling que sigue convierte la equivariancia en invariancia (local). Toma una imagen de 4 × 4 con un borde vertical en su mitad superior y el kernel de borde vertical \( \mathbf{K} \):

\[ \mathbf{I} = \begin{pmatrix} 1 & 1 & 5 & 5 \\ 1 & 1 & 5 & 5 \\ 1 & 1 & 1 & 1 \\ 1 & 1 & 1 & 1 \end{pmatrix}, \qquad \mathbf{K} = \begin{pmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{pmatrix} \]

La salida "valid" es de 2 × 2. En la posición (1,1) la ventana es el bloque de 3 × 3 superior izquierdo; cada una de sus filas aporta \( -1 \cdot (\text{left}) + 0 + 1 \cdot (\text{right}) \): las filas 1 y 2 dan \( -1 + 5 = 4 \) cada una, la fila 3 da \( -1 + 1 = 0 \), total \( 8 \). En (1,2) la ventana se mueve una columna a la derecha: las filas 1 y 2 dan \( -1 + 5 = 4 \), la fila 3 da \( 0 \), total \( 8 \). En (2,1) la ventana cubre las filas 2–4: la fila 2 da \( 4 \), las filas 3 y 4 dan \( 0 \), total \( 4 \); en (2,2), igualmente, \( 4 \). Así, \( \mathbf{I} \star \mathbf{K} = \left(\begin{smallmatrix} 8 & 8 \\ 4 & 4 \end{smallmatrix}\right) \). Con el kernel transpuesto (un detector de bordes horizontales), la posición (1,1) da \( -(1+1+5) + (1+1+1) = -4 \) y la posición (1,2) da \( -(1+5+5) + 3 = -8 \); la segunda fila los repite. Una **ReLU** lleva a cero las respuestas negativas, de modo que esta imagen excita el mapa de bordes verticales y no el horizontal; un **max-pooling de 2 × 2** del mapa vertical devuelve el único número 8: "hay un borde vertical en algún lugar de esta imagen", sin importar dónde. El bloque de código confirma los números.



In [ ]:
conv_valid <- function(I, K){ k <- nrow(K); H <- nrow(I); W <- ncol(I)
  outer(1:(H-k+1), 1:(W-k+1), Vectorize(function(r,c) sum(K * I[r:(r+k-1), c:(c+k-1)]))) }
maxpool <- function(M, s=2){ H <- nrow(M)/s; W <- ncol(M)/s
  outer(1:H,1:W, Vectorize(function(r,c) max(M[((r-1)*s+1):(r*s),((c-1)*s+1):(c*s)]))) }
Iv <- matrix(c(1,1,5,5, 1,1,5,5, 1,1,1,1, 1,1,1,1), 4, 4, byrow=TRUE)
Kv <- matrix(c(-1,0,1, -1,0,1, -1,0,1), 3, 3, byrow=TRUE)
mv <- conv_valid(Iv, Kv); mh <- conv_valid(Iv, t(Kv))
cat("vertical-edge map:\n"); print(mv); cat("horizontal-edge map:\n"); print(mh)
cat("after ReLU + 2x2 max-pool -> vertical:", as.vector(maxpool(pmax(mv,0))),
    " horizontal:", as.vector(maxpool(pmax(mh,0))), "\n")



El código reproduce el resultado a mano: \( \left(\begin{smallmatrix} 8 & 8 \\ 4 & 4 \end{smallmatrix}\right) \) para el kernel vertical y \( \left(\begin{smallmatrix} -4 & -8 \\ -4 & -8 \end{smallmatrix}\right) \) para el horizontal. Tras la ReLU y el pooling, la imagen queda resumida en dos números, 8 y 0: "un borde vertical, ningún borde horizontal".

### 6.3 Mapas de características y pooling en imágenes reales

El bloque de código aplica un pequeño **banco de filtros fijos** —detectores de bordes de Sobel vertical y horizontal, un laplaciano (responde a puntos y líneas finas), un desenfoque gaussiano y un detector de "manchas" (blobs) por diferencia de gaussianas ajustado al tamaño de los nódulos simulados— a un glifo y a un parche con nódulo, con relleno "same", y muestra los mapas antes y después de un max-pooling de 4 × 4.



In [ ]:
FILTERS <- list(
  `Sobel v`=matrix(c(-1,0,1,-2,0,2,-1,0,1),3,3,byrow=TRUE),
  `Sobel h`=matrix(c(-1,-2,-1,0,0,0,1,2,1),3,3,byrow=TRUE),
  Laplacian=matrix(c(0,-1,0,-1,4,-1,0,-1,0),3,3,byrow=TRUE),
  blur=gkern(1.0,3), blob=gkern(1.2,9)-gkern(2.4,9))
feature_maps <- function(img) lapply(FILTERS, function(K) pmax(conv_same(img,K),0))
set.seed(7); gl <- draw_glyph("diagL"); pt <- nod$X[which(nod$y==1)[1],,]
par(mfrow=c(4,6), mar=c(0.4,0.4,1.4,0.4), oma=c(0,0,2,0))
for (ex in list(list(gl,"glyph"), list(pt,"nodule patch"))){ maps <- feature_maps(ex[[1]])
  image(t(ex[[1]][nrow(ex[[1]]):1,]),col=grey.colors(32,1,0),axes=FALSE,main=ex[[2]],cex.main=0.85); box()   # fila: original + mapas
  for (nm in names(FILTERS)){ M <- maps[[nm]]; image(t(M[nrow(M):1,]),col=hcl.colors(32,"Inferno"),axes=FALSE,main=nm,cex.main=0.85); box() }
  plot.new()                                                                                                  # fila: vacía + mapas tras pooling
  for (nm in names(FILTERS)){ Mp <- maxpool(maps[[nm]],4); image(t(Mp[nrow(Mp):1,]),col=hcl.colors(32,"Inferno"),axes=FALSE,main="pooled",cex.main=0.8); box() } }
mtext("Feature maps (ReLU of the correlation) and, below each, its 4x4 max-pool", outer=TRUE, line=0.3, cex=0.9, font=2)
cat("map sizes: patch 16x16 -> pooled", paste(dim(maxpool(feature_maps(pt)[[1]],4)), collapse=" x "), "\n")



<p class="caption"><b>Figura 15.</b> Columna 1 (en cada bloque): los originales; las columnas restantes son los cinco mapas de características y, bajo cada uno, su versión tras un max-pooling de 4 × 4, para un glifo (las dos primeras filas) y un parche con nódulo (las dos últimas filas). Los mapas de Sobel se iluminan en los bordes según su orientación; el laplaciano responde a estructuras finas y al ruido; el mapa de desenfoque es una copia suavizada; el detector de manchas responde con fuerza y de forma selectiva al nódulo y débilmente a la textura de fondo: es una plantilla de "un objeto brillante y redondo de aproximadamente este tamaño" aplicada en todas las posiciones. El pooling descarta la posición exacta dentro de cada bloque de 4 × 4, pero conserva la información de que la característica estaba presente: un nódulo desplazado uno o dos píxeles cae en la misma celda agrupada. Apilar varias veces convolución, no linealidad y pooling, con los kernels <em>aprendidos</em> por retropropagación en lugar de fijos, es la red neuronal convolucional de LeCun y colaboradores,<sup class="c"><a href="#r14">[14]</a></sup> y la que ganó ImageNet en 2012.<sup class="c"><a href="#r16">[16]</a></sup></p>

### 6.4 Una etapa convolucional inicial mínima, y el regreso de la robustez ante desplazamientos

El bloque de código ensambla las piezas en una **etapa convolucional inicial de filtros fijos**: el banco de filtros, una ReLU, un max-pooling de 4 × 4 y el máximo y la media globales de cada mapa, seguidos de un perceptrón sobre el vector resultante. Nada en la etapa inicial se aprende —los filtros son los diseñados a mano de más arriba—, así que esto no es una red convolucional en sentido pleno; es una demostración de que la *estructura* de la convolución y el pooling, y no solo el aprendizaje de los filtros, es lo que aporta la robustez. El experimento repite la prueba de desplazamiento para cuatro sistemas: el perceptrón con píxeles crudos; el perceptrón con píxeles crudos entrenado con aumento de datos por desplazamiento; la etapa convolucional inicial; y la etapa inicial con aumento de datos.



In [ ]:
conv_features <- function(X, pool=4){ t(apply(X, 1, function(v){ maps <- feature_maps(matrix(v,SG,SG))
  c(unlist(lapply(maps, function(M) as.vector(maxpool(M,pool)))),
    sapply(maps, max), sapply(maps, mean)) })) }
aug_shifts <- list(c(1,0),c(-1,0),c(0,1),c(0,-1))
Xaug <- rbind(G$X, do.call(rbind, lapply(aug_shifts, function(s) shift_flat(G$X,s[1],s[2])))); yaug <- rep(G$y, 5)
fit_eval <- function(Xtr,ytr,featfun){ Z <- featfun(Xtr); ce <- colMeans(Z); s <- apply(Z,2,sd); s[s==0] <- 1
  fit <- nnet(scale(Z,ce,s), class.ind(ytr), size=16, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
  function(Xte) mean(predict(fit, scale(featfun(Xte),ce,s), type="class")==Gte$y) }
idf <- function(X) X
systems <- list(`raw pixels`=fit_eval(G$X,G$y,idf),
                `raw + shift aug`=fit_eval(Xaug,yaug,idf),
                `conv front end`=fit_eval(G$X,G$y,conv_features),
                `conv + aug`=fit_eval(Xaug,yaug,conv_features))
tab <- sapply(systems, function(ev) sapply(shifts[c(1,2,5,7)], function(s) round(ev(shift_flat(Gte$X,s[1],s[2])),3)))
rownames(tab) <- sapply(shifts[c(1,2,5,7)], function(s) sprintf("shift (%d,%d)",s[1],s[2]))
cat("features per image: raw", ncol(G$X), "| conv front end", ncol(conv_features(G$X[1:2,])), "\n")
print(t(tab))



Lee la tabla fila por fila. El **perceptrón con píxeles crudos** decae rápidamente con el desplazamiento. El **aumento de datos** repara principalmente los desplazamientos que se le mostraron —un píxel en cada dirección— y ayuda menos con desplazamientos mayores: la red memorizó unas pocas posiciones adicionales; no aprendió que la posición es irrelevante. La **etapa convolucional inicial** sola, con filtros fijos y una única etapa de pooling, es más robusta, porque el pooling de 4 × 4 tolera un desplazamiento mientras la característica permanezca en el mismo bloque. La **combinación** es la más robusta con los desplazamientos mayores: la representación agrupada hace que un glifo desplazado se *vea parecido* para la red, de modo que la invariancia aprendida a partir de ejemplos pequeños se extrapola a desplazamientos que nunca vio. Esa extrapolación es justamente lo que se llama un **sesgo inductivo** —un supuesto incorporado en la arquitectura que permite al modelo generalizar más allá de los casos de la muestra—, y es la razón por la que las redes convolucionales aprendieron a partir de miles de radiografías lo que los perceptrones no podían aprender a partir de millones de píxeles.

<div class="box key"><h4>De esta etapa inicial al aprendizaje profundo, y al aprendizaje por transferencia</h4><p>Una red convolucional real difiere del bloque de código anterior en tres aspectos. Sus filtros se <em>aprenden</em> por retropropagación junto con el clasificador, de modo que el detector de manchas que ajustamos a mano se descubre a partir de los datos, y también detectores que a nadie se le ocurrió diseñar. Apila varios bloques de convolución–ReLU–pooling, de modo que las capas posteriores combinan bordes en partes y partes en objetos. Y se entrena con muchas más imágenes de las que tiene un hospital; por eso el <strong>aprendizaje por transferencia</strong> (transfer learning) es la receta estándar en imágenes médicas: a una red preentrenada con los 1.2 millones de imágenes naturales de ImageNet<sup class="c"><a href="#r16">[16]</a></sup> se le reemplaza la última capa y se la ajusta finamente (fine-tuning) con unos pocos miles de imágenes médicas. Shin y colaboradores mostraron que los filtros aprendidos con fotografías se transfieren de manera útil a la CT, porque los bordes, las manchas y las texturas son universales;<sup class="c"><a href="#r17">[17]</a></sup> todos los artículos fundamentales de la Sección 8 lo usaron. Este notebook no ejecuta una red profunda, porque el curso no usa ningún framework de aprendizaje profundo; MedMNIST<sup class="c"><a href="#r8">[8]</a></sup> con un framework como Keras o Torch es el lugar para hacerlo después.</p></div>

### 6.5 La misma lección con los parches simulados

Los parches permiten un experimento complementario que los glifos no permiten: la **posición de la lesión** es un parámetro del generador. El bloque de código genera tres conjuntos de datos de nódulos en los que el centro de la lesión se desplaza aleatoriamente hasta 1, 3 y 6 píxeles alrededor del centro del parche de 16 × 16, y evalúa con validación cruzada el mismo perceptrón con tres representaciones: los píxeles crudos, las 14 características diseñadas a mano de la Sección 3 y la etapa convolucional inicial con el detector de manchas.



In [ ]:
conv_feat_patch <- function(X, pool=4){ t(apply(X, 1, function(v){ img <- matrix(v,SIZE,SIZE)
  maps <- lapply(FILTERS, function(K) pmax(conv_same(img,K),0))
  c(sapply(maps, max), sapply(maps, mean), unlist(lapply(maps, function(M) as.vector(maxpool(M,pool))))) })) }
jitters <- c(1,3,6); reps <- list()
for (j in jitters){ set.seed(20+j); pj <- make_patches(500, "nodule", jitter=j)
  Pp <- flat_patch(pj$X); Fp <- t(sapply(1:500, function(k) extract_features(pj$X[k,,]))); Cp <- conv_feat_patch(pj$X)
  fo <- strat_folds(pj$y, 3)
  for (rep in c("raw pixels","14 features","conv front end")){ Xin <- switch(rep, `raw pixels`=Pp, `14 features`=Fp, Cp)
    p <- mlp_bin_cv(Xin, pj$y, 4, fo, 0.5)
    a <- sapply(1:3, function(f){ te <- fo==f; mean((p[te]>0.5)==pj$y[te]) })
    reps[[rep]] <- rbind(reps[[rep]], c(mean(a), sd(a))) } }
cols <- c(palette_neg, palette_acc, palette_pos)
plot(NA, xlim=range(jitters), ylim=c(0.5,1), xlab="max lesion displacement (pixels)", ylab="CV accuracy",
     main="Nodule task: three representations vs lesion position"); i <- 1
for (rep in names(reps)){ m <- reps[[rep]]; lines(jitters, m[,1], type="b", pch=19, col=cols[i])
  arrows(jitters, m[,1]-m[,2], jitters, m[,1]+m[,2], angle=90, code=3, length=0.03, col=cols[i]); i <- i+1 }
legend("bottomleft", names(reps), col=cols, lwd=2, pch=19, bty="n", cex=0.8)
for (rep in names(reps)) cat(sprintf("%-16s %s\n", rep, paste(sprintf("%.3f+/-%.3f", reps[[rep]][,1], reps[[rep]][,2]), collapse="  ")))



<p class="caption"><b>Figura 16.</b> Exactitud de validación cruzada del mismo perceptrón con tres representaciones de los parches con nódulos, a medida que se permite que la lesión se aleje más del centro. Con píxeles crudos, la red se degrada a medida que el nódulo puede estar en cualquier lugar: tiene que aprender una plantilla distinta para cada posición, y quinientas imágenes no bastan. Las características diseñadas a mano son invariantes por construcción —el área, el contraste y la textura no dependen de dónde esté la mancha— y su curva es la más plana. La etapa convolucional inicial queda entre las otras dos: su detector de manchas se aplica en todas partes y su pooling descarta la posición dentro de bloques de 4 × 4, de modo que se degrada de forma gradual; una red con filtros aprendidos y más etapas de pooling sería aún más plana. La figura resume todo el argumento de la sección en una sola imagen: con pocos datos, la invariancia debe venir de algún lugar distinto de los datos, ya sea de características diseñadas o de la arquitectura.</p>

<div class="exercise"><h4>Ejercicio 4</h4><p>Reemplaza <code>pool=4</code> por <code>pool=16</code> en <code>conv_feat_patch</code>, de modo que cada mapa se reduzca a un único máximo y una única media globales. La representación es ahora completamente invariante a la traslación; ¿sigue siendo discriminativa? Luego pasa el conjunto de microcalcificaciones (<code>mc</code>) por las mismas tres representaciones. ¿Qué filtro del banco transporta la señal para esa tarea, y por qué fallan en ella las características de primer orden?</p></div>

## 7. Qué puede salir mal en la clasificación de imágenes médicas

### 7.1 Aprendizaje por atajos: la red aprende el hospital

La Unidad 1 simuló un identificador de equipo que predecía la etiqueta en el hospital de entrenamiento y no en otros lugares. Con imágenes, el factor de confusión no necesita ser una columna aparte: puede estar *en los píxeles*. Zech y colaboradores entrenaron una red convolucional para detectar neumonía en radiografías reunidas de varios hospitales y encontraron que identificaba el hospital de origen a partir de la imagen con una exactitud casi perfecta —a partir de marcadores de lateralidad, texto incrustado, el campo de visión— y usaba la prevalencia de neumonía del hospital como predictor, de modo que su rendimiento externo quedaba muy por debajo de su validación cruzada interna.<sup class="c"><a href="#r18">[18]</a></sup> Geirhos y colaboradores llamaron a este fenómeno **aprendizaje por atajos** (shortcut learning): el modelo encuentra la regularidad más fácil que predice la etiqueta en la distribución de entrenamiento, sea o no la buscada.<sup class="c"><a href="#r19">[19]</a></sup> DeGrave y colaboradores mostraron que los detectores de COVID-19 en radiografías de 2020 —con positivos de un conjunto de fuentes y negativos de otro— se apoyaban en marcadores, bordes de la imagen y la posición de los hombros en lugar de en los pulmones.<sup class="c"><a href="#r20">[20]</a></sup> El bloque de código reproduce el mecanismo con el hospital A, cuyos parches positivos llevan un marcador en la esquina el 90 % de las veces, y evalúa el perceptrón con parches nuevos del mismo hospital y de un hospital B sin marcadores.



In [ ]:
fit_pix <- function(X, y){ ce <- colMeans(X); s <- apply(X,2,sd); s[s==0] <- 1
  m <- nnet(scale(X,ce,s), y, size=6, entropy=TRUE, maxit=250, trace=FALSE, decay=0.5, MaxNWts=1e5)
  function(Xn) as.vector(predict(m, scale(Xn,ce,s))) }
set.seed(10); XA <- make_patches(1000, "nodule", hospital="A", marker_rate=0.9)   # hospital de entrenamiento
set.seed(11); XA2 <- make_patches(400, "nodule", hospital="A", marker_rate=0.9)   # casos nuevos, mismo hospital
set.seed(12); XB <- make_patches(400, "nodule", hospital="B")                     # otro hospital, sin marcadores
set.seed(13); XH <- make_patches(1000, "nodule", hospital="B")                    # un conjunto de entrenamiento sin marcadores
mA <- fit_pix(flat_patch(XA$X), XA$y); mH <- fit_pix(flat_patch(XH$X), XH$y)
ev <- function(m, d) mean((m(flat_patch(d$X))>0.5)==d$y)
res <- rbind(`trained at hospital A (marker)` = c(`training`=ev(mA,XA), `new, same hospital`=ev(mA,XA2), `new, hospital B`=ev(mA,XB)),
             `trained without markers`         = c(ev(mH,XH), ev(mH,XA2), ev(mH,XB)))
cat(sprintf("fraction of hospital-A positives carrying the marker: %.2f\n",
            mean(apply(XA$X[XA$y==1,,], 1, function(im) all(im[2:3,(SIZE-3):(SIZE-1)]>0.9)))))
print(round(res,3))



El modelo entrenado en el hospital A se ve excelente en casa —su exactitud con casos nuevos del mismo hospital está muy por encima de la del modelo sin marcadores, porque el marcador es una pista mucho más fácil que un nódulo tenue— y pierde muchos puntos en el hospital B, donde debe apoyarse en lo que sea que haya aprendido sobre los nódulos, que es menos de lo que aprendió el modelo sin marcadores. Como en la Unidad 1, ningún número interno advirtió de la falla: la exactitud de entrenamiento y la exactitud en el mismo hospital eran ambas altas. La causa puede hacerse visible, y ese es el tema de la sección siguiente.

### 7.2 ¿Dónde está mirando la red? Sensibilidad a la oclusión

Un **mapa de saliencia** muestra qué partes de una imagen impulsan una predicción. El más simple, introducido por Zeiler y Fergus, es la **sensibilidad a la oclusión**: cubre un pequeño cuadrado de la imagen con un valor neutro, registra cuánto baja la probabilidad predicha de la clase, mueve el cuadrado y repite; el mapa de caídas muestra dónde está la evidencia.<sup class="c"><a href="#r21">[21]</a></sup> No requiere más que la capacidad de obtener la salida de probabilidades del modelo, por lo que funciona tanto para un perceptrón como para una red profunda. La caída se mide en log-odds (logaritmo de la razón de probabilidades), \( \log[p/(1-p)] \), en lugar de en probabilidad, porque un modelo confiado entrega \( p \approx 1 \) para muchos parches, y una caída de 0.999 a 0.99 —un cambio de diez veces en las odds— sería invisible en la escala de probabilidad. El bloque de código la calcula para los dos modelos de la Sección 7.1 en tres parches positivos del hospital A que llevan el marcador.



In [ ]:
logit <- function(p){ p <- pmin(pmax(p,1e-6),1-1e-6); log(p/(1-p)) }
occ_map <- function(model, img, size=5, step=2){ base <- model(matrix(as.vector(t(img)),1))
  H <- matrix(0,SIZE,SIZE); N <- matrix(0,SIZE,SIZE)
  for (r in seq(1,SIZE-size+1,step)) for (c in seq(1,SIZE-size+1,step)){ oc <- img; oc[r:(r+size-1),c:(c+size-1)] <- mean(img)
    d <- logit(base) - logit(model(matrix(as.vector(t(oc)),1)))
    H[r:(r+size-1),c:(c+size-1)] <- H[r:(r+size-1),c:(c+size-1)] + d; N[r:(r+size-1),c:(c+size-1)] <- N[r:(r+size-1),c:(c+size-1)] + 1 }
  list(map=H/pmax(N,1), p=base) }
idx <- which(XA2$y==1 & apply(XA2$X, 1, function(im) im[2,SIZE-2]>0.9))[1:3]
par(mfrow=c(3,3), mar=c(1,1,2.6,1), oma=c(0,0,2.4,0))
for (i in idx){ om_s <- occ_map(mA, XA2$X[i,,]); om_h <- occ_map(mH, XA2$X[i,,])
  image(t(XA2$X[i,,][SIZE:1,]), col=grey.colors(32,1,0), axes=FALSE, zlim=c(0,1))
  title(sprintf("p=%.2f (A-model)\n%.2f (marker-free)", om_s$p, om_h$p), cex.main=0.85); box()
  for (om in list(list(om_s,"hospital-A model"), list(om_h,"marker-free model"))){ v <- max(abs(om[[1]]$map))+1e-9
    image(t(om[[1]]$map[SIZE:1,]), col=colorRampPalette(c(palette_neg,"white",palette_pos))(32), axes=FALSE, zlim=c(-v,v))
    title(sprintf("%s\nmax drop %.1f", om[[2]], max(om[[1]]$map)), cex.main=0.85); box() } }
mtext("Occlusion sensitivity (orange: covering this region lowers the log-odds of 'nodule')", outer=TRUE, line=0.4, cex=0.9, font=2)



<p class="caption"><b>Figura 17.</b> Tres parches positivos del hospital A (izquierda) y los mapas de sensibilidad a la oclusión de los dos modelos. Para el modelo entrenado en el hospital A (centro), la evidencia es el marcador de la esquina: cubrir la esquina superior derecha hace colapsar la probabilidad, y cubrir el nódulo cambia poco. Para el modelo sin marcadores (derecha), la evidencia es el propio nódulo. Los mapas son gruesos —el oclusor es de 5 × 5 píxeles y se mueve en pasos de 2—, pero habrían revelado el atajo antes del despliegue, que es como se descubrieron los marcadores y los bordes de los detectores de COVID-19.<sup class="c"><a href="#r20">[20]</a></sup> Los mapas de saliencia tienen sus propios límites: muestran <em>dónde</em> miró el modelo, no <em>por qué</em>, pueden ser ruidosos, y un mapa que resalta la lesión no prueba que el modelo la haya entendido. Usados como herramienta de depuración, como aquí, son indispensables.</p>

<div class="box warn"><h4>El atajo no tiene por qué ser un marcador</h4><p>Cualquier variable que se correlacione con la etiqueta en la muestra de entrenamiento y sea visible en los píxeles puede ser un atajo: el modelo de equipo (a través de la textura del ruido), el servicio (las radiografías portátiles se toman a pacientes más graves), la presencia de un drenaje pleural en imágenes de neumotórax —un caso documentado por Oakden-Rayner y colaboradores, cuya red detectaba el tratamiento en lugar de la afección<sup class="c"><a href="#r22">[22]</a></sup>— o la regla que se coloca junto a las lesiones que los dermatólogos ya consideran sospechosas. La defensa es la que dio la Unidad 1: evaluar con datos de otro centro, estratificar la evaluación por cada variable que pudiera ser un atajo y mirar los mapas de saliencia de los errores <em>y</em> de los aciertos confiados.</p></div>

### 7.3 Desbalance de clases, prevalencia y ruido en las etiquetas

Dos propiedades de los conjuntos de datos de imágenes reales que las simulaciones balanceadas ocultan merecen un experimento cada una. **Desbalance de clases**: en el tamizaje los positivos son raros, y un clasificador entrenado con un 95 % de negativos aprende que "negativo" es una buena respuesta; su exactitud es alta y su sensibilidad es baja. El remedio más simple suele ser mantener el modelo y **mover el umbral**, como se hizo en la Sección 4.5. **Ruido en las etiquetas**: las etiquetas de los grandes conjuntos de datos de radiografías se extrajeron de los informes mediante minería de texto y son erróneas en una fracción conocida de los casos;<sup class="c"><a href="#r3">[3]</a></sup> la revisión de Frénay y Verleysen examina lo que el ruido hace a los clasificadores.<sup class="c"><a href="#r23">[23]</a></sup> Ocurren dos cosas: el modelo aprende peor y —de manera más insidiosa— la exactitud *medida* en un conjunto de prueba con etiquetas ruidosas queda limitada por debajo de la exactitud real, de modo que un modelo perfecto parece imperfecto.



In [ ]:
set.seed(30); Xneg <- make_patches(950, "nodule", positive=FALSE); set.seed(31); Xpos <- make_patches(50, "nodule", positive=TRUE)
Xi <- rbind(flat_patch(Xneg$X), flat_patch(Xpos$X)); Fi <- rbind(t(sapply(1:950,function(k)extract_features(Xneg$X[k,,]))),
                                                                 t(sapply(1:50,function(k)extract_features(Xpos$X[k,,]))))
yi <- c(rep(0,950), rep(1,50)); set.seed(0); foi <- strat_folds(yi, 5)
pi_oof <- mlp_bin_cv(Fi, yi, 6, foi, 0.5)
roc_i <- roc(yi, pi_oof, quiet=TRUE, direction="<")
t90 <- coords(roc_i, x=0.90, input="sensitivity", ret="threshold")[1,1]
metrics <- function(pred) c(accuracy=mean(pred==yi), sensitivity=mean(pred[yi==1]), specificity=1-mean(pred[yi==0]), `positives called`=sum(pred))
res_i <- rbind(`always negative`=metrics(rep(0,length(yi))), `MLP, threshold 0.5`=metrics(as.integer(pi_oof>=0.5)),
               `MLP, threshold for 90% sens`=metrics(as.integer(pi_oof>=t90)))
cat(sprintf("prevalence %.1f%% | ROC-AUC %.3f | threshold for 90%% sensitivity %.2f\n", 100*mean(yi), as.numeric(auc(roc_i)), t90))
print(round(res_i,3))



La regla trivial que declara todo negativo tiene una exactitud de 0.95 —la prevalencia de negativos— y una sensibilidad de cero. El perceptrón con las características diseñadas a mano lo hace mejor que eso con el umbral por defecto y aun así no detecta una parte de los nódulos, porque con 50 positivos entre 1,000 la manera más barata de reducir la pérdida de entrenamiento es ser conservador; el ROC-AUC muestra que sus probabilidades ordenan bien los casos. Bajar el umbral hasta el valor que alcanza un 90 % de sensibilidad recupera la mayoría de los casos no detectados, a costa de más llamados positivos y de una exactitud *menor*. En una tarea desbalanceada, la exactitud no mide nada útil (Suplemento S1); la sensibilidad, la especificidad, la curva ROC y, para el clínico, el PPV con la prevalencia del despliegue sí lo hacen.



In [ ]:
set.seed(40); tr <- sample(nrow(G$X), round(0.7*nrow(G$X))); levs <- GLYPHS
flip_labels <- function(y, q){ y <- as.character(y); fl <- runif(length(y))<q
  y[fl] <- sapply(which(fl), function(i) sample(setdiff(levs, y[i]), 1)); factor(y, levels=levs) }
qs <- c(0,0.1,0.2,0.3,0.4); clean_acc <- noisy_acc <- numeric(length(qs))
for (k in seq_along(qs)){ set.seed(40+k); ytr <- flip_labels(G$y[tr], qs[k]); yte_n <- flip_labels(G$y[-tr], qs[k])
  fit <- nnet(Gs[tr,], class.ind(ytr), size=24, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
  pr <- predict(fit, Gs[-tr,], type="class"); clean_acc[k] <- mean(pr==G$y[-tr]); noisy_acc[k] <- mean(pr==yte_n) }
plot(qs, clean_acc, type="b", pch=19, col=palette_neg, ylim=c(0.3,1.02), xlab="fraction of flipped training labels",
     ylab="test accuracy", main="Label noise on the glyphs")
lines(qs, noisy_acc, type="b", pch=15, lty=2, col=palette_pos); lines(qs, 1-qs, lty=3, col="grey40")
legend("bottomleft", c("vs CLEAN labels","vs equally NOISY labels","ceiling 1 - q"),
       col=c(palette_neg,palette_pos,"grey40"), lty=c(1,2,3), pch=c(19,15,NA), bty="n", cex=0.75)
cat("clean-label accuracy:", round(clean_acc,3), "| noisy-label accuracy:", round(noisy_acc,3), "\n")



<p class="caption"><b>Figura 18.</b> Exactitud de prueba del perceptrón de glifos a medida que crece la fracción de etiquetas de entrenamiento invertidas, medida frente a etiquetas de prueba limpias (azul) y frente a etiquetas de prueba corrompidas con la misma tasa (naranja). La curva azul se degrada gradualmente, porque las etiquetas erróneas, repartidas entre las demás clases, se cancelan en parte; con más imágenes de entrenamiento la curva sería más plana. La curva naranja es la que realmente vería un investigador con etiquetas ruidosas: corre por debajo del techo gris \( 1-q \) aproximadamente en el error real del modelo, de modo que un modelo que en realidad es exacto se <em>reporta</em> como mucho peor cuando el ruido es alto. El ruido <em>sistemático</em> —una regla de minería de informes que siempre pasa por alto un tipo de neumonía— no se promedia y es mucho más dañino; las etiquetas de incertidumbre de CheXpert<sup class="c"><a href="#r4">[4]</a></sup> y los conjuntos de prueba adjudicados por radiólogos de los artículos fundamentales existen por esta razón.</p>

### 7.4 Validación externa, estudios con lectores y la situación regulatoria de la IA en imágenes

Tres prácticas separan un modelo que funciona en un notebook de un dispositivo que funciona en un hospital. La **validación externa** evalúa el modelo congelado con datos de instituciones, equipos y períodos que no aportaron nada a su desarrollo; la lista de verificación CLAIM para la IA en imágenes médicas la convierte en un requisito de reporte, junto con la muestra, el estándar de referencia, el preprocesamiento, el procedimiento de selección del modelo y métricas con intervalos de confianza.<sup class="c"><a href="#r24">[24]</a></sup> Un **estudio con lectores** (reader study) compara el modelo con clínicos que leen los mismos casos en las mismas condiciones, idealmente también leyendo *con* el modelo, porque el sistema desplegado es la combinación. La **evaluación prospectiva** mide lo que les ocurre a los pacientes cuando se usa el modelo.

El registro regulatorio muestra con qué poca frecuencia se hacen estas cosas. Benjamens y colaboradores catalogaron 64 dispositivos basados en IA autorizados o aprobados por la Food and Drug Administration de los Estados Unidos hasta 2020, la mayoría en radiología, y encontraron que la mayor parte de los resúmenes públicos revelaba poco sobre la evaluación.<sup class="c"><a href="#r25">[25]</a></sup> Wu y colaboradores examinaron 130 de estos dispositivos: 126 se habían evaluado solo de manera retrospectiva, 93 no reportaban una evaluación multicéntrica y pocos reportaban el rendimiento por subgrupo demográfico.<sup class="c"><a href="#r26">[26]</a></sup> La mayoría son **software como dispositivo médico** (software as a medical device) autorizado por la vía 510(k) por equivalencia con un dispositivo existente, no mediante ensayos clínicos; el marco de la Unidad 1 —la clasificación de alto riesgo de la Ley de IA europea (AI Act), los principios de buenas prácticas de aprendizaje automático de la FDA, los seis principios de la OMS— es el contexto en el que las decisiones del ingeniero en esta unidad se convierten en obligaciones legales.

## 8. Lectura de los artículos fundamentales

Los artículos siguientes definieron el campo entre 2016 y 2020 y son los que con mayor probabilidad citarán tus presentaciones del Taller 2. Para cada uno se plantean las preguntas de la Unidad 1: cuáles fueron los objetos, las etiquetas y su estándar de referencia, la muestra, la métrica, el comparador, y qué falta. La tabla es una guía de lectura, no un sustituto de los artículos.

| Artículo | Tarea y datos | Resultado principal | Comparador | Qué preguntar |
|---|---|---|---|---|
| Gulshan et al., <em>JAMA</em> 2016<sup class="c"><a href="#r27">[27]</a></sup> | Retinopatía diabética derivable en 128,175 fotografías de retina, cada una graduada por varios de 54 oftalmólogos; validado en dos conjuntos de 9,963 y 1,748 imágenes | AUC de alrededor de 0.99 en ambos; en el punto de operación de alta sensibilidad, sensibilidad de alrededor del 97 % y especificidad de alrededor del 93 % | Voto de mayoría de los oftalmólogos como estándar de referencia | Retrospectivo; imágenes de las cámaras y poblaciones de los centros de entrenamiento; se excluyeron las imágenes no graduables; sin desenlaces de los pacientes |
| Esteva et al., <em>Nature</em> 2017<sup class="c"><a href="#r28">[28]</a></sup> | Lesiones cutáneas: una red preentrenada con ImageNet y ajustada finamente con 129,450 imágenes clínicas de 2,032 enfermedades; probada con imágenes confirmadas por biopsia | Sensibilidad–especificidad en o por encima de los puntos de operación de 21 dermatólogos en tres tareas binarias | 21 dermatólogos certificados leyendo las mismas imágenes | Los dermatólogos leyeron fotografías, no pacientes; las imágenes de prueba fueron seleccionadas; tonos de piel subrepresentados |
| Rajpurkar et al., 2017 (CheXNet)<sup class="c"><a href="#r29">[29]</a></sup> | Neumonía en ChestX-ray14; red convolucional de 121 capas; conjunto de prueba de 420 imágenes etiquetadas por cuatro radiólogos | Puntaje F1 superior al promedio de los cuatro radiólogos | Radiólogos leyendo las mismas imágenes frontales sin historia clínica | Etiquetas de entrenamiento extraídas de informes con errores conocidos;<sup class="c"><a href="#r3">[3]</a></sup> una sola institución; intervalos de confianza amplios con 420 imágenes |
| McKinney et al., <em>Nature</em> 2020<sup class="c"><a href="#r30">[30]</a></sup> | Mamografía de tamizaje de cáncer de mama: 25,856 mujeres en el Reino Unido y 3,097 en EE. UU., con el estado de cáncer obtenido del seguimiento | Menos falsos positivos y falsos negativos que los lectores originales; superó a seis radiólogos en un estudio con lectores | Decisiones históricas del primer lector; un estudio con lectores de seis radiólogos | Haibe-Kains et al. respondieron que el código, el modelo y los datos no se compartieron, por lo que el resultado no puede reproducirse ni auditarse;<sup class="c"><a href="#r31">[31]</a></sup> retrospectivo |
| Ardila et al., <em>Nat Med</em> 2019<sup class="c"><a href="#r32">[32]</a></sup> | Tamizaje de cáncer de pulmón con CT de baja dosis del National Lung Screening Trial: red 3-D sobre volúmenes completos; cáncer confirmado por biopsia | AUC de alrededor de 0.94 en 6,716 casos; menos falsos positivos y falsos negativos que seis radiólogos cuando no se disponía de una CT previa | Seis radiólogos, con y sin el examen previo | Datos de ensayo de un solo país y una sola época; validación independiente pequeña; sin uso prospectivo |
| Liu et al., <em>Lancet Digit Health</em> 2019<sup class="c"><a href="#r33">[33]</a></sup> | Revisión sistemática del aprendizaje profundo frente a profesionales de la salud en imágenes médicas: 82 de más de 30,000 estudios cumplieron los criterios de inclusión | Sensibilidad agrupada de alrededor del 87 % frente al 86 % y especificidad de alrededor del 93 % frente al 91 % en los pocos estudios con una comparación justa | Clínicos, en los 14 estudios que compararon a ambos en la misma muestra externa | Solo 25 de los 82 estudios hicieron validación externa; pocos reportaron intervalos de confianza; la base de evidencia es deficiente |

Tres patrones recorren la tabla. El estándar de referencia es un juicio humano o una biopsia, nunca las propias etiquetas de entrenamiento del modelo, y los artículos más creíbles usaron varios lectores y una adjudicación. El comparador es casi siempre un grupo de radiólogos que leen imágenes *sin* el contexto clínico que tendrían en la práctica, lo que favorece a la máquina. Y ninguno de los artículos originales fue prospectivo: la afirmación más fuerte que cualquiera de ellos respalda es "en imágenes retrospectivas de estas fuentes, el punto de operación del modelo es comparable al de estos lectores", que es donde la revisión de Topol de 2019 situó la evidencia.<sup class="c"><a href="#r34">[34]</a></sup> El vocabulario de este curso —muestra, etiqueta, fuga de datos, validación externa, prevalencia, intervalo de confianza, atajo— basta para leer críticamente cada uno de estos artículos.

<div class="box bridge"><h4>Puente hacia la Unidad 9</h4><p>La Unidad 9 vuelve a estos sistemas desde el lado del despliegue: cómo un clasificador se convierte en un sistema de diagnóstico con una interfaz de usuario, un flujo de trabajo y un plan de monitoreo; qué agrega a las imágenes la minería de datos de la ficha clínica electrónica; y las preguntas de ética profesional que las secciones 7.1–7.4 plantearon en forma técnica. Los experimentos de esta unidad —la grilla de topología, la prueba de desplazamiento, el atajo y su mapa de oclusión— son la evidencia que se te pedirá presentar en esa discusión.</p></div>

## 9. Resumen

Ahora deberías poder explicar, sin apuntes:

- Qué codifica el valor de un píxel en radiografía, CT (unidades Hounsfield), MRI, ultrasonido, histopatología, dermatoscopía y fotografía de retina; por qué los valores de CT son comparables entre equipos y los de MRI no; qué son el ventaneo, la profundidad de bits y DICOM.
- Por qué una imagen es a la vez una matriz y un vector de características, qué descarta el aplanamiento (la adyacencia) y las cinco etapas del pipeline de imágenes.
- Qué hacen el escalamiento min–max, la estandarización, la ecualización del histograma (y el algoritmo que la sustenta), el redimensionamiento, la eliminación de ruido gaussiana y el aumento de datos, y la condición de preservación de la etiqueta que rige el aumento.
- Cómo calcular características de primer orden, de textura GLCM y de forma; cómo construir a mano una matriz de co-ocurrencia y su contraste, homogeneidad y energía; por qué las características diseñadas superan a los píxeles crudos con mil imágenes.
- Cómo clasifica un perceptrón imágenes aplanadas, cómo estimar su exactitud con validación cruzada estratificada repetida, cómo leer su matriz de confusión y sus plantillas de primera capa, y cómo elegir un umbral para una prevalencia de tamizaje.
- Las cuatro reglas de un experimento de topología honesto, qué mostraron la grilla profundidad × ancho, las curvas de aprendizaje y los experimentos de penalización, activación y optimizador, y por qué se necesita validación cruzada anidada para reportar el resultado.
- Por qué un perceptrón no es invariante a la traslación, cómo desarrollar a mano una convolución, una ReLU y un max-pooling, y por qué convolución más pooling más aumento de datos generaliza a desplazamientos a los que el aumento por sí solo no generaliza.
- Qué es el aprendizaje por atajos, cómo reproducirlo con un marcador de hospital, cómo lo revela un mapa de sensibilidad a la oclusión, y por qué la validación externa, la evaluación estratificada y la inspección de saliencia son las defensas.
- Por qué la exactitud carece de sentido bajo desbalance de clases, qué le hace el ruido en las etiquetas a un modelo y a su exactitud medida, y qué dice el registro regulatorio sobre cómo se ha evaluado la IA en imágenes.
- Cómo leer un artículo fundamental: objetos, etiquetas, estándar de referencia, muestra, métrica, comparador y qué falta.

<div class="box warn"><h4>Qué se simula aquí, y qué hizo distinto la edición en Python</h4><p>R no incluye ningún conjunto de datos de dígitos manuscritos ni de imágenes médicas, así que esta unidad construye sus dos fuentes de imágenes a partir de código: un conjunto de glifos de 8 × 8 con 6 clases en lugar de los dígitos manuscritos de la edición en Python, y parches de 16 × 16 similares a radiografías en lugar de sus parches de 32 × 32. Por lo tanto, los números difieren de los del notebook de Python, pero todas las conclusiones cualitativas son las mismas. Una diferencia de lenguaje da forma a la Sección 5: el <code>nnet</code> de R ofrece una única capa oculta logística optimizada por BFGS, de modo que la profundidad se estudió con <code>neuralnet</code> (sobre una representación PCA, para facilitar la convergencia) y el experimento de tasa de aprendizaje, momento y tamaño de lote se realizó con un entrenador de descenso de gradiente escrito desde cero con activación lineal rectificada. La unión de estas tres herramientas cubre el terreno que cubría el único <code>MLPClassifier</code> de Python.</p></div>

## 10. Autoevaluación

<details><summary>1. Un vóxel de CT tiene el valor −850 y un vóxel de MRI tiene el valor 850. ¿Qué puedes decir sobre el tejido en cada caso, y por qué la respuesta es de naturaleza distinta?</summary>El valor de CT está en unidades Hounsfield, una escala física: −850 es cercano al aire, así que el vóxel es muy probablemente pulmón aireado. El valor de MRI está en unidades arbitrarias que dependen del equipo, la bobina y la secuencia de pulsos; por sí solo no dice nada, y solo su relación con otros vóxeles de la misma adquisición es informativa. Por eso la MRI requiere normalización de intensidades por imagen y la CT usa ventaneo.</details>

<details><summary>2. Desarrolla la ecualización del histograma de una imagen de 2 × 2 con valores (0, 0, 1, 3) en una escala de 4 niveles.</summary>Histograma: el nivel 0 aparece dos veces, el nivel 1 una vez, el nivel 2 nunca, el nivel 3 una vez. Distribución acumulada F: F(0) = 2/4 = 0.5, F(1) = 0.75, F(2) = 0.75, F(3) = 1. El nuevo valor en [0,1] es el propio F (o reescalado al rango de niveles). Los dos píxeles oscuros pasan a 0.5, el 1 pasa a 0.75 y el 3 a 1: los valores de la imagen se desplazan hacia arriba y se separan, y el contraste entre los dos píxeles oscuros desaparece; la ecualización de una imagen diminuta es burda.</details>

<details><summary>3. En el bloque de código del aumento de datos, ¿cuáles de las seis transformaciones preservan la etiqueta de una radiografía de tórax clasificada por neumonía? ¿Y de una mamografía clasificada por lateralidad (mama izquierda/derecha)? ¿Y de una imagen dermatoscópica clasificada por melanoma?</summary>Neumonía: las seis preservan la etiqueta, aunque conviene evitar el volteo izquierda–derecha si el modelo también informará el lado. Lateralidad: el volteo izquierda–derecha destruye la etiqueta (es la etiqueta), las demás la preservan. Melanoma: las seis, porque una lesión cutánea no tiene una orientación ni una exposición canónicas.</details>

<details><summary>4. Calcula el contraste GLCM de la imagen de 3 × 3 [[0,1,0],[0,1,0],[0,1,0]] con 2 niveles y el desplazamiento "un píxel a la derecha".</summary>Pares horizontales: cada fila da (0,1) y (1,0), seis pares en total, todos con |i − j| = 1. P(0,1) = P(1,0) = 1/2; contraste = Σ (i − j)² P = 1 · (1/2 + 1/2) = 1. Homogeneidad = Σ P/(1 + |i − j|) = 1/2; energía = 1/4 + 1/4 = 1/2. Un patrón de franjas verticales tiene el máximo contraste horizontal.</details>

<details><summary>5. El perceptrón de la Sección 4.1 obtuvo una exactitud de (por ejemplo) 0.93 ± 0.02 sobre diez particiones. La red de un colega obtiene 0.95 en una única partición 70/30 de los mismos datos. ¿Es mejor?</summary>No de manera demostrable. Las diez particiones de nuestro modelo abarcaron varios puntos porcentuales, de modo que 0.95 está dentro de la dispersión de una sola partición del mismo modelo. Una comparación justa entrena ambos con las mismas particiones repetidas y compara las diferencias pareadas entre pliegues; y si el colega seleccionó la arquitectura en esa partición, el 0.95 está sesgado de forma optimista (Sección 5.6).</details>

<details><summary>6. En la grilla profundidad × ancho, el salto decisivo fue el que va del modelo lineal a una capa oculta, mientras que agregar más capas cambió la exactitud solo dentro del ruido entre pliegues. ¿Por qué la profundidad adicional debería dar una ganancia confiable tan pequeña con este tamaño de muestra?</summary>Una sola capa oculta ya hace separables las clases, de modo que captura la mayor parte de la señal disponible y eleva la exactitud muy por encima de la línea base lineal. Cada capa adicional agrega capacidad, pero con solo unos pocos cientos de imágenes esa capacidad no puede ser restringida por los datos, así que cualquier ganancia es pequeña e incierta (comparable a la desviación estándar entre pliegues), y la optimización es más difícil porque el gradiente debe atravesar más capas. Lo "profundo" ayuda cuando los datos son lo bastante grandes para restringir los parámetros adicionales; con una muestra pequeña, el experimento no puede demostrar un beneficio de la profundidad.</details>

<details><summary>7. ¿Por qué la mejor celda de una grilla evaluada con validación cruzada no es una estimación insesgada del rendimiento del modelo elegido, y qué estimación debería reportarse en su lugar?</summary>La mejor celda se eligió porque era el máximo de varias estimaciones ruidosas en los mismos pliegues; el máximo de cantidades ruidosas está sesgado hacia arriba. La estimación de la validación cruzada anidada —la exactitud en los pliegues externos del procedimiento "elegir por CV interna y luego ajustar"— es insesgada para el procedimiento desplegado y es la que debería reportarse.</details>

<details><summary>8. Calcula a mano la correlación cruzada "valid" de [[2,2,2],[2,2,2],[0,0,0]] con el kernel [[1,1,1],[0,0,0],[−1,−1,−1]], y luego el resultado tras la ReLU.</summary>El kernel es de 3 × 3 y la imagen de 3 × 3, de modo que la salida es un solo número: 1·(2+2+2) + 0·(2+2+2) − 1·(0+0+0) = 6. ReLU(6) = 6. El kernel detecta "claro arriba, oscuro abajo" y la imagen es exactamente eso, por lo que la respuesta es máxima. Con el kernel invertido verticalmente, la respuesta sería −6 y la ReLU daría 0.</details>

<details><summary>9. En la tabla de desplazamientos de la Sección 6.4, el aumento con desplazamientos de un píxel reparó los desplazamientos pequeños pero no los mayores, mientras que la etapa convolucional inicial con aumento fue robusta a ambos. Explica la diferencia con una oración para cada caso.</summary>El aumento de datos enseñó de memoria al perceptrón con píxeles crudos unas pocas posiciones adicionales, y un desplazamiento mayor es una posición que nunca vio; la representación convolucional agrupada hace que las imágenes desplazadas sean <em>similares como vectores</em>, de modo que lo que la red aprendió de los ejemplos pequeños se transfiere a desplazamientos mayores: la arquitectura aporta un sesgo inductivo que los datos por sí solos no aportaban.</details>

<details><summary>10. El modelo entrenado en el hospital A tuvo mayor exactitud en el mismo hospital que el modelo sin marcadores y, sin embargo, es el peor modelo. ¿Qué evaluación única lo habría mostrado, y qué diagnóstico habría mostrado por qué?</summary>La evaluación con parches de otro hospital sin el marcador (validación externa) mostró la caída; el mapa de sensibilidad a la oclusión mostró que la evidencia usada era el marcador de la esquina y no el nódulo.</details>

<details><summary>11. Un clasificador para una tarea con 5 % de prevalencia reporta una exactitud del 95 %. ¿Qué es lo mínimo que podría estar haciendo, y qué dos números pedirías?</summary>Declarar negativos todos los casos logra un 95 % de exactitud con sensibilidad cero. Pide la sensibilidad y la especificidad (o la curva ROC), y luego calcula el PPV a la prevalencia del despliegue; la exactitud no es informativa bajo desbalance.</details>

<details><summary>12. En el experimento de ruido en las etiquetas, ¿qué curva observaría un investigador con datos de etiquetas ruidosas, y qué implica eso para comparar un modelo con radiólogos en un conjunto de prueba así?</summary>La curva naranja, medida frente a etiquetas de prueba ruidosas, que queda acotada cerca de 1 − q. Un modelo puede parecer mucho peor de lo que es, y una comparación con radiólogos sobre las mismas etiquetas ruidosas queda distorsionada en una dirección que depende de si los errores de los radiólogos se correlacionan con el ruido de las etiquetas. El remedio es un conjunto de prueba con un estándar de referencia adjudicado, como el que usaron los artículos fundamentales.</details>

## Glosario

<dl class="gloss">
<dt>Aumento de datos (augmentation)</dt><dd>Ampliar la muestra de entrenamiento con copias transformadas de sus imágenes (volteos, rotaciones, desplazamientos, cambios de intensidad) con la condición de que la transformación preserve la etiqueta; se aplica solo a los pliegues de entrenamiento.</dd>
<dt>Convolución o correlación cruzada (convolution, cross-correlation)</dt><dd>Deslizar un kernel pequeño sobre una imagen y registrar, en cada posición, la suma de los productos de los pesos del kernel por los píxeles subyacentes; la salida es un mapa de características. Las bibliotecas de aprendizaje profundo calculan la correlación cruzada y la llaman convolución.</dd>
<dt>Red neuronal convolucional (convolutional neural network, CNN)</dt><dd>Una red cuyas primeras capas son convoluciones aprendidas seguidas de no linealidades y pooling, de modo que los mismos detectores de características se aplican en todas las posiciones; la arquitectura de LeCun (1998) y de AlexNet (2012).</dd>
<dt>DICOM</dt><dd>Digital Imaging and Communications in Medicine: el formato estándar de archivo y de comunicación de las imágenes médicas, cuyo encabezado almacena los parámetros de adquisición y los datos del paciente.</dd>
<dt>Equivariancia / invariancia a la traslación (translation equivariance / invariance)</dt><dd>Una representación es equivariante si desplazar la entrada desplaza la representación de forma correspondiente, e invariante si la representación no cambia; la convolución es equivariante, y el pooling la hace localmente invariante.</dd>
<dt>Mapa de características (feature map)</dt><dd>La salida de un kernel aplicado a toda la imagen; grande donde la imagen se parece localmente al kernel.</dd>
<dt>Matriz de co-ocurrencia de niveles de gris (grey-level co-occurrence matrix, GLCM)</dt><dd>El descriptor de textura de Haralick: los conteos normalizados de pares de niveles de gris a un desplazamiento dado, a partir de los cuales se calculan el contraste, la homogeneidad, la energía y la correlación.</dd>
<dt>Ecualización del histograma (histogram equalisation)</dt><dd>Mapeo no lineal de intensidades a través de la distribución acumulada del histograma de la imagen, que distribuye los niveles de gris por todo el rango disponible.</dd>
<dt>Unidad Hounsfield (Hounsfield unit, HU)</dt><dd>La escala de intensidad de la CT: 1000 (μ − μ<sub>water</sub>)/(μ<sub>water</sub> − μ<sub>air</sub>), de modo que el agua vale 0 y el aire −1000; una magnitud física comparable entre equipos.</dd>
<dt>Sesgo inductivo (inductive bias)</dt><dd>Los supuestos incorporados en la arquitectura o el entrenamiento de un modelo que le permiten generalizar más allá de los casos exactos de la muestra; la convolución y el pooling codifican el supuesto de que la clase no depende de la posición.</dd>
<dt>Ruido en las etiquetas (label noise)</dt><dd>Errores en las etiquetas de entrenamiento o de prueba; el ruido uniforme degrada un modelo de forma leve, pero acota la exactitud medida en alrededor de 1 − q; el ruido sistemático se aprende como si fuera señal.</dd>
<dt>Curva de aprendizaje (learning curve)</dt><dd>Exactitud de validación cruzada y de entrenamiento graficadas frente al número de ejemplos de entrenamiento; el diagnóstico para decidir si se necesitan más datos o un modelo distinto.</dd>
<dt>Validación cruzada anidada (nested cross-validation)</dt><dd>Una validación cruzada externa que estima el rendimiento de un procedimiento en el que la configuración del modelo se elige mediante una validación cruzada interna realizada solo con los pliegues de entrenamiento.</dd>
<dt>Sensibilidad a la oclusión (occlusion sensitivity)</dt><dd>Un mapa de saliencia que se obtiene cubriendo regiones sucesivas de la imagen y registrando la caída de la probabilidad predicha de la clase.</dd>
<dt>Pooling (max-pooling)</dt><dd>Reemplazar cada bloque de un mapa de características por su máximo (o su media), lo que reduce el tamaño del mapa y descarta la posición exacta de la característica dentro del bloque.</dd>
<dt>Radiómica (radiomics)</dt><dd>El cálculo de grandes conjuntos estandarizados de características de primer orden, de textura y de forma a partir de lesiones segmentadas, seguido de aprendizaje automático clásico.</dd>
<dt>Estudio con lectores (reader study)</dt><dd>Una evaluación en la que clínicos y el modelo leen los mismos casos en condiciones comparables; el comparador estándar en los artículos fundamentales.</dd>
<dt>Aprendizaje por atajos (shortcut learning)</dt><dd>La dependencia de un modelo respecto de una regularidad espuria de los datos de entrenamiento —un marcador, un borde, la textura de un equipo— que predice la etiqueta allí, pero no en otros lugares.</dd>
<dt>Plantilla o pesos de la primera capa (template, first-layer weights)</dt><dd>El vector de pesos de una unidad oculta reorganizado al tamaño de la imagen; muestra el patrón de píxeles al que la unidad responde con más fuerza.</dd>
<dt>Aprendizaje por transferencia (transfer learning)</dt><dd>Reutilizar una red preentrenada con un gran conjunto de datos (ImageNet) reemplazando su última capa y ajustándola finamente con un conjunto de datos médicos más pequeño.</dd>
<dt>Ventaneo (windowing)</dt><dd>Mapear un intervalo elegido de valores de CT a la escala de grises visualizada, recortando los valores que quedan fuera de él; distintas ventanas muestran pulmón, tejido blando o hueso en el mismo corte.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Litjens G, Kooi T, Bejnordi BE, et al. A survey on deep learning in medical image analysis. <em>Med Image Anal</em>. 2017;42:60–88. <a href="https://doi.org/10.1016/j.media.2017.07.005">doi:10.1016/j.media.2017.07.005</a></li>
<li id="r2">Hounsfield GN. Computerized transverse axial scanning (tomography): Part 1. Description of system. <em>Br J Radiol</em>. 1973;46(552):1016–1022. <a href="https://doi.org/10.1259/0007-1285-46-552-1016">doi:10.1259/0007-1285-46-552-1016</a></li>
<li id="r3">Wang X, Peng Y, Lu L, Lu Z, Bagheri M, Summers RM. ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases. <em>Proc IEEE CVPR</em>. 2017:3462–3471. <a href="https://doi.org/10.1109/CVPR.2017.369">doi:10.1109/CVPR.2017.369</a></li>
<li id="r4">Irvin J, Rajpurkar P, Ko M, et al. CheXpert: a large chest radiograph dataset with uncertainty labels and expert comparison. <em>Proc AAAI Conf Artif Intell</em>. 2019;33(1):590–597. <a href="https://doi.org/10.1609/aaai.v33i01.3301590">doi:10.1609/aaai.v33i01.3301590</a></li>
<li id="r5">Johnson AEW, Pollard TJ, Berkowitz SJ, et al. MIMIC-CXR, a de-identified publicly available database of chest radiographs with free-text reports. <em>Sci Data</em>. 2019;6:317. <a href="https://doi.org/10.1038/s41597-019-0322-0">doi:10.1038/s41597-019-0322-0</a></li>
<li id="r6">Lee RS, Gimenez F, Hoogi A, Miyake KK, Gorovoy M, Rubin DL. A curated mammography data set for use in computer-aided detection and diagnosis research. <em>Sci Data</em>. 2017;4:170177. <a href="https://doi.org/10.1038/sdata.2017.177">doi:10.1038/sdata.2017.177</a></li>
<li id="r7">Tschandl P, Rosendahl C, Kittler H. The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. <em>Sci Data</em>. 2018;5:180161. <a href="https://doi.org/10.1038/sdata.2018.161">doi:10.1038/sdata.2018.161</a></li>
<li id="r8">Yang J, Shi R, Wei D, et al. MedMNIST v2 — a large-scale lightweight benchmark for 2D and 3D biomedical image classification. <em>Sci Data</em>. 2023;10:41. <a href="https://doi.org/10.1038/s41597-022-01721-8">doi:10.1038/s41597-022-01721-8</a></li>
<li id="r9">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r10">Alpaydin E, Kaynak C. Optical Recognition of Handwritten Digits. UCI Machine Learning Repository; 1998. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r11">Setio AAA, Traverso A, de Bel T, et al. Validation, comparison, and combination of algorithms for automatic detection of pulmonary nodules in computed tomography images: the LUNA16 challenge. <em>Med Image Anal</em>. 2017;42:1–13. <a href="https://doi.org/10.1016/j.media.2017.06.015">doi:10.1016/j.media.2017.06.015</a></li>
<li id="r12">Shorten C, Khoshgoftaar TM. A survey on image data augmentation for deep learning. <em>J Big Data</em>. 2019;6:60. <a href="https://doi.org/10.1186/s40537-019-0197-0">doi:10.1186/s40537-019-0197-0</a></li>
<li id="r13">Haralick RM, Shanmugam K, Dinstein I. Textural features for image classification. <em>IEEE Trans Syst Man Cybern</em>. 1973;SMC-3(6):610–621. <a href="https://doi.org/10.1109/TSMC.1973.4309314">doi:10.1109/TSMC.1973.4309314</a></li>
<li id="r14">LeCun Y, Bottou L, Bengio Y, Haffner P. Gradient-based learning applied to document recognition. <em>Proc IEEE</em>. 1998;86(11):2278–2324. <a href="https://doi.org/10.1109/5.726791">doi:10.1109/5.726791</a></li>
<li id="r15">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r16">Krizhevsky A, Sutskever I, Hinton GE. ImageNet classification with deep convolutional neural networks. <em>Commun ACM</em>. 2017;60(6):84–90. <a href="https://doi.org/10.1145/3065386">doi:10.1145/3065386</a></li>
<li id="r17">Shin H-C, Roth HR, Gao M, et al. Deep convolutional neural networks for computer-aided detection: CNN architectures, dataset characteristics and transfer learning. <em>IEEE Trans Med Imaging</em>. 2016;35(5):1285–1298. <a href="https://doi.org/10.1109/TMI.2016.2528162">doi:10.1109/TMI.2016.2528162</a></li>
<li id="r18">Zech JR, Badgeley MA, Liu M, Costa AB, Titano JJ, Oermann EK. Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. <em>PLoS Med</em>. 2018;15(11):e1002683. <a href="https://doi.org/10.1371/journal.pmed.1002683">doi:10.1371/journal.pmed.1002683</a></li>
<li id="r19">Geirhos R, Jacobsen J-H, Michaelis C, et al. Shortcut learning in deep neural networks. <em>Nat Mach Intell</em>. 2020;2:665–673. <a href="https://doi.org/10.1038/s42256-020-00257-z">doi:10.1038/s42256-020-00257-z</a></li>
<li id="r20">DeGrave AJ, Janizek JD, Lee S-I. AI for radiographic COVID-19 detection selects shortcuts over signal. <em>Nat Mach Intell</em>. 2021;3:610–619. <a href="https://doi.org/10.1038/s42256-021-00338-7">doi:10.1038/s42256-021-00338-7</a></li>
<li id="r21">Zeiler MD, Fergus R. Visualizing and understanding convolutional networks. In: <em>Computer Vision — ECCV 2014</em>. Lecture Notes in Computer Science, vol 8689. Springer; 2014:818–833. <a href="https://doi.org/10.1007/978-3-319-10590-1_53">doi:10.1007/978-3-319-10590-1_53</a></li>
<li id="r22">Oakden-Rayner L, Dunnmon J, Carneiro G, Ré C. Hidden stratification causes clinically meaningful failures in machine learning for medical imaging. <em>Proc ACM Conf Health Inference Learn (CHIL)</em>. 2020:151–159. <a href="https://doi.org/10.1145/3368555.3384468">doi:10.1145/3368555.3384468</a></li>
<li id="r23">Frénay B, Verleysen M. Classification in the presence of label noise: a survey. <em>IEEE Trans Neural Netw Learn Syst</em>. 2014;25(5):845–869. <a href="https://doi.org/10.1109/TNNLS.2013.2292894">doi:10.1109/TNNLS.2013.2292894</a></li>
<li id="r24">Mongan J, Moy L, Kahn CE Jr. Checklist for Artificial Intelligence in Medical Imaging (CLAIM): a guide for authors and reviewers. <em>Radiol Artif Intell</em>. 2020;2(2):e200029. <a href="https://doi.org/10.1148/ryai.2020200029">doi:10.1148/ryai.2020200029</a></li>
<li id="r25">Benjamens S, Dhunnoo P, Meskó B. The state of artificial intelligence-based FDA-approved medical devices and algorithms: an online database. <em>npj Digit Med</em>. 2020;3:118. <a href="https://doi.org/10.1038/s41746-020-00324-0">doi:10.1038/s41746-020-00324-0</a></li>
<li id="r26">Wu E, Wu K, Daneshjou R, Ouyang D, Ho DE, Zou J. How medical AI devices are evaluated: limitations and recommendations from an analysis of FDA approvals. <em>Nat Med</em>. 2021;27:582–584. <a href="https://doi.org/10.1038/s41591-021-01312-x">doi:10.1038/s41591-021-01312-x</a></li>
<li id="r27">Gulshan V, Peng L, Coram M, et al. Development and validation of a deep learning algorithm for detection of diabetic retinopathy in retinal fundus photographs. <em>JAMA</em>. 2016;316(22):2402–2410. <a href="https://doi.org/10.1001/jama.2016.17216">doi:10.1001/jama.2016.17216</a></li>
<li id="r28">Esteva A, Kuprel B, Novoa RA, et al. Dermatologist-level classification of skin cancer with deep neural networks. <em>Nature</em>. 2017;542:115–118. <a href="https://doi.org/10.1038/nature21056">doi:10.1038/nature21056</a></li>
<li id="r29">Rajpurkar P, Irvin J, Zhu K, et al. CheXNet: radiologist-level pneumonia detection on chest X-rays with deep learning. <em>arXiv</em>. 2017;1711.05225. <a href="https://doi.org/10.48550/arXiv.1711.05225">doi:10.48550/arXiv.1711.05225</a> (preprint; sin revisión por pares)</li>
<li id="r30">McKinney SM, Sieniek M, Godbole V, et al. International evaluation of an AI system for breast cancer screening. <em>Nature</em>. 2020;577:89–94. <a href="https://doi.org/10.1038/s41586-019-1799-6">doi:10.1038/s41586-019-1799-6</a></li>
<li id="r31">Haibe-Kains B, Adam GA, Hosny A, et al. Transparency and reproducibility in artificial intelligence. <em>Nature</em>. 2020;586:E14–E16. <a href="https://doi.org/10.1038/s41586-020-2766-y">doi:10.1038/s41586-020-2766-y</a></li>
<li id="r32">Ardila D, Kiraly AP, Bharadwaj S, et al. End-to-end lung cancer screening with three-dimensional deep learning on low-dose chest computed tomography. <em>Nat Med</em>. 2019;25:954–961. <a href="https://doi.org/10.1038/s41591-019-0447-x">doi:10.1038/s41591-019-0447-x</a></li>
<li id="r33">Liu X, Faes L, Kale AU, et al. A comparison of deep learning performance against health-care professionals in detecting diseases from medical imaging: a systematic review and meta-analysis. <em>Lancet Digit Health</em>. 2019;1(6):e271–e297. <a href="https://doi.org/10.1016/S2589-7500(19)30123-2">doi:10.1016/S2589-7500(19)30123-2</a></li>
<li id="r34">Topol EJ. High-performance medicine: the convergence of human and artificial intelligence. <em>Nat Med</em>. 2019;25:44–56. <a href="https://doi.org/10.1038/s41591-018-0300-7">doi:10.1038/s41591-018-0300-7</a></li>
<li id="r35">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a> (la referencia del perceptrón de una sola capa oculta de <code>nnet</code> usado aquí)</li>
<li id="r36">Fritsch S, Guenther F, Wright MN. neuralnet: Training of Neural Networks. R package. <a href="https://doi.org/10.32614/CRAN.package.neuralnet">doi:10.32614/CRAN.package.neuralnet</a></li>
<li id="r37">Meyer D, Dimitriadou E, Hornik K, Weingessel A, Leisch F. e1071: Misc Functions of the Department of Statistics, Probability Theory Group (Formerly E1071), TU Wien. R package. <a href="https://doi.org/10.32614/CRAN.package.e1071">doi:10.32614/CRAN.package.e1071</a></li>
<li id="r38">Liaw A, Wiener M. randomForest: Breiman and Cutler's random forests for classification and regression. R package. <a href="https://doi.org/10.32614/CRAN.package.randomForest">doi:10.32614/CRAN.package.randomForest</a></li>
<li id="r39">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — El complemento en R de todo el curso; PDF gratuito en statlearning.com.</li>
<li>Goodfellow I, Bengio Y, Courville A. <em>Deep Learning</em>. MIT Press; 2016. — El Capítulo 9 (redes convolucionales) es la referencia para la Sección 6; el Capítulo 7 para la regularización y la detención temprana; el Capítulo 8 para los optimizadores de la Sección 5.5. Disponible gratis en línea en deeplearningbook.org.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — Capítulo 5 para el perceptrón multicapa, incluida la discusión sobre invariancia y convolución.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Capítulo 7 para la evaluación de modelos y el sesgo de la selección de modelos; Capítulo 11 para las redes neuronales. PDF gratuito.</li>
<li>Gonzalez RC, Woods RE. <em>Digital Image Processing</em>. 4th ed. Pearson; 2018. — Capítulos 3 y 10 para la ecualización del histograma, el filtrado y la segmentación.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>CRAN Task View: Machine Learning &amp; Statistical Learning — <a href="https://cran.r-project.org/view=MachineLearning">cran.r-project.org/view=MachineLearning</a>; el catálogo de paquetes de R para los métodos de esta unidad.</li>
<li>MedMNIST — <a href="https://medmnist.com">medmnist.com</a>; conjuntos de datos de imágenes médicas pequeños y estandarizados; el paso siguiente recomendado después de este notebook (con un framework de aprendizaje profundo).</li>
<li>ISIC Archive — <a href="https://www.isic-archive.com">isic-archive.com</a>; imágenes dermatoscópicas públicas y los desafíos anuales de clasificación de melanoma.</li>
<li>Estándar DICOM — <a href="https://www.dicomstandard.org">dicomstandard.org</a>; el paquete de R <code>oro.dicom</code> lee archivos DICOM.</li>
<li>Lista de verificación CLAIM — <a href="https://pubs.rsna.org/page/ai/claim">pubs.rsna.org/page/ai/claim</a>; la lista de verificación de reporte de la Sección 7.4 en su versión vigente.</li>
</ul>
